# AffectFusion — Day 3 Complete Pipeline (v3 — Full 3,000-Row Gold Test Verification): Zero-Leakage 80/10/10 Split, 3K Gold Verification, Gemini Flash Train-Only Augmentation & Feature Engineering

**Project:** *AffectFusion: A Cross-Attention Transformer Framework for Explainable Multimodal Emotion Recognition from Text and Emoji Semantics*  
**Input (Day 2 Drive Output):** `/content/drive/MyDrive/AffectFusion_Day2_Gemini/04_filtered_and_audit/day2_gemini_labeled_conf_ge3.parquet` (`44,580` verified rows)  
**Output Workspace:** `/content/drive/MyDrive/AffectFusion_Day3/`  
**Runtime Required:** **Colab Free Tier CPU Runtime** *(Zero GPU needed — save your free T4 GPU quota for Day 4 `xlm-roberta-base` training!)*

---

### What Is Included in v3
1. **Strict Zero-Leakage 80/10/10 Split (`Stage 1`) with `Fear` Banglish Safeguard:**
   - Locks **Val (`3,000`)** and **Test (`3,000`)** strictly from **100% real scraped rows** (quarantining all `twitter_dair_*` supplemental rows to Train only).
   - Proportional language safeguard prevents `Fear` `BNG` starvation in Train when the scraped pool is tight.
2. **Full 3,000-Row Gold Test Set Manual Verification (`Stage 2A` + `Stage 2B`):**
   - Splits **100% of your locked `test_3k` (`3,000` rows)** evenly across both teammates (**1,500 rows each**: `gold_member1_ragib_1500rows.xlsx` & `gold_member2_galib_1500rows.xlsx`), balanced per class (`167`/`166`) with blinded AI labels and Excel dropdown validation.
   - **Incremental Cohen's Kappa ($\kappa$) Evaluator:** Score progress anytime (e.g., at 500, 1,500, or all 3,000 rows).
   - **Automatic Gold Test Upgrade (`APPLY_GOLD_LABELS_TO_TEST = True`):** Once all 3,000 rows are verified, automatically upgrades `test_3k_locked.parquet` with your **100% human-verified ground truth** (while preserving `ai_label_pre_gold` for auditing), so re-running Stage 4 produces a true Gold `test_3k.csv`!
3. **Gemini Flash Train-Only Emoji-Preserving Augmentation (`Stage 3A` + `Stage 3B`):**
   - Uses **Gemini Flash** (~143 API calls = ~6–11 minutes on CPU) to paraphrase the `3,528` Train-only deficit rows (`Fear`, `Surprise`, `Anger`, `Disgust`, `Anticipation`) while preserving original emojis, target emotion, and cross-lingual fluency (`ENG`/`BN`/`BNG`), with automatic local lexical fallback if no API key is provided or quota runs out.
4. **Low-RAM (< 600 MB) Streaming Feature Engineering & Drive Checkpointing (`Stage 4`):**
   - Checkpoints every stage (and every class in Stage 3) to Google Drive, and exports `train_24k.csv`, `val_3k.csv`, `test_3k.csv`, `.parquet` copies, and `label_map.json` in 5,000-row streaming chunks.


## Stage 0: Mount Google Drive, Install Dependencies & Load Configuration
Run all 3 cells in Stage 0 at the start of **every** Colab session (standard **CPU** runtime).


In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

DAY2_DIR = '/content/drive/MyDrive/AffectFusion_Day2_Gemini'
DAY3_DIR = '/content/drive/MyDrive/AffectFusion_Day3'

DIRS = {
    'day2_audit': os.path.join(DAY2_DIR, '04_filtered_and_audit'),
    'day2_csv':   os.path.join(DAY2_DIR, '05_final_csv_exports'),
    'base':       DAY3_DIR,
    'splits':     os.path.join(DAY3_DIR, '01_locked_splits'),
    'gold':       os.path.join(DAY3_DIR, '02_gold_verification'),
    'aug':        os.path.join(DAY3_DIR, '03_train_augmented'),
    'final':      os.path.join(DAY3_DIR, '04_feature_engineered_final'),
    'logs':       os.path.join(DAY3_DIR, 'logs'),
}

for k, d in DIRS.items():
    if not k.startswith('day2_'):
        os.makedirs(d, exist_ok=True)

DAY2_PARQUET = os.path.join(DIRS['day2_audit'], 'day2_gemini_labeled_conf_ge3.parquet')
DAY2_CSV     = os.path.join(DIRS['day2_csv'],   'day2_gemini_final_labeled_conf_ge3.csv')

assert os.path.exists(DAY2_PARQUET) or os.path.exists(DAY2_CSV), (
    f"Day 2 labeled dataset not found in {DAY2_DIR}! Make sure Stage 3 of Day 2 finished."
)

print("Google Drive mounted.")
print("  Day 2 Input Parquet:", DAY2_PARQUET, "| exists:", os.path.exists(DAY2_PARQUET))
print("  Day 2 Input CSV    :", DAY2_CSV,     "| exists:", os.path.exists(DAY2_CSV))
print("  Day 3 Workspace    :", DAY3_DIR)


Mounted at /content/drive
Google Drive mounted.
  Day 2 Input Parquet: /content/drive/MyDrive/AffectFusion_Day2_Gemini/04_filtered_and_audit/day2_gemini_labeled_conf_ge3.parquet | exists: True
  Day 2 Input CSV    : /content/drive/MyDrive/AffectFusion_Day2_Gemini/05_final_csv_exports/day2_gemini_final_labeled_conf_ge3.csv | exists: True
  Day 3 Workspace    : /content/drive/MyDrive/AffectFusion_Day3


In [2]:
# Colab-safe installs (no -U -> preserves Colab's pandas/requests versions; zero pip conflicts)
!pip install -q google-genai pydantic emoji openpyxl scikit-learn


In [3]:
import re
import gc
import json
import time
import math
import glob
import random
import hashlib
from datetime import datetime, timezone
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import emoji
from tqdm.auto import tqdm

# ==============================================================================
# DAY 3 MASTER CONFIGURATION (EXACT 30,000 ROWS | 80/10/10 SPLIT | 3K GOLD TEST)
# ==============================================================================
CLASSES = [
    "Joy",          # 0
    "Neutral",      # 1
    "Anger",        # 2
    "Sadness",      # 3
    "Surprise",     # 4
    "Sarcasm",      # 5
    "Fear",         # 6
    "Disgust",      # 7
    "Anticipation", # 8
]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}
ID2LABEL = {i: c for i, c in enumerate(CLASSES)}
LANG2ID  = {"ENG": 0, "BN": 1, "BNG": 2}

# Exact per-class targets summing to EXACTLY 24,000 Train / 3,000 Val / 3,000 Test (30,000 Total):
EXTRA_VT_CLASSES = {"Joy", "Neutral", "Sadness"}
SPLIT_TARGETS = {
    c: {
        "train": 2666 if c in EXTRA_VT_CLASSES else 2667,
        "val":    334 if c in EXTRA_VT_CLASSES else  333,
        "test":   334 if c in EXTRA_VT_CLASSES else  333,
    }
    for c in CLASSES
}

assert sum(v["train"] for v in SPLIT_TARGETS.values()) == 24_000
assert sum(v["val"]   for v in SPLIT_TARGETS.values()) ==  3_000
assert sum(v["test"]  for v in SPLIT_TARGETS.values()) ==  3_000

CONFIG = {
    "RANDOM_SEED": 42,
    "CHUNK_SIZE": 5000,
    "MAX_CHAR_LEN": 600,
    "LANG_RATIOS": {"ENG": 0.40, "BN": 0.30, "BNG": 0.30},
    "EMOJI_TARGET_RATIO": 0.70,
    "GOLD_PER_CLASS": None,             # None = 100% of locked test_3k (3,000 rows: 1,500 per member)
    "AUG_BATCH_SIZE": 25,               # 25 rows/call -> ~143 Gemini calls for all 3,528 aug rows

    # Checkpointed Drive paths
    "VAL_LOCKED_PQ":     os.path.join(DIRS['splits'], "val_3k_locked.parquet"),
    "TEST_LOCKED_PQ":    os.path.join(DIRS['splits'], "test_3k_locked.parquet"),
    "TRAIN_PRE_AUG_PQ":  os.path.join(DIRS['splits'], "train_pre_aug.parquet"),
    "SPLIT_REPORT_JSON": os.path.join(DIRS['logs'],   "stage1_split_report.json"),

    # Full 3,000-Row Gold Verification paths (1,500 rows each)
    "GOLD_M1_XLSX":      os.path.join(DIRS['gold'],   "gold_member1_ragib_1500rows.xlsx"),
    "GOLD_M2_XLSX":      os.path.join(DIRS['gold'],   "gold_member2_galib_1500rows.xlsx"),
    "GOLD_KEY_PQ":       os.path.join(DIRS['gold'],   "gold_3000_ai_key_internal.parquet"),
    "GOLD_MERGED_CSV":   os.path.join(DIRS['gold'],   "gold_3000_verified_merged.csv"),
    "KAPPA_REPORT_JSON": os.path.join(DIRS['logs'],   "stage2_cohens_kappa_3000_report.json"),

    "TRAIN_24K_AUG_PQ":  os.path.join(DIRS['aug'],    "train_24k_balanced.parquet"),
    "AUG_REPORT_JSON":   os.path.join(DIRS['logs'],   "stage3_augmentation_report.json"),

    "FINAL_TRAIN_CSV":   os.path.join(DIRS['final'],  "train_24k.csv"),
    "FINAL_VAL_CSV":     os.path.join(DIRS['final'],  "val_3k.csv"),
    "FINAL_TEST_CSV":    os.path.join(DIRS['final'],  "test_3k.csv"),
    "FINAL_TRAIN_PQ":    os.path.join(DIRS['final'],  "train_24k.parquet"),
    "FINAL_VAL_PQ":      os.path.join(DIRS['final'],  "val_3k.parquet"),
    "FINAL_TEST_PQ":     os.path.join(DIRS['final'],  "test_3k.parquet"),
    "LABEL_MAP_JSON":    os.path.join(DIRS['final'],  "label_map.json"),
    "FINAL_SUMMARY_JS":  os.path.join(DIRS['logs'],   "stage4_final_dataset_summary.json"),
}

random.seed(CONFIG["RANDOM_SEED"])
np.random.seed(CONFIG["RANDOM_SEED"])

print("Day 3 Master Configuration Loaded (v3 — Full 3,000-Row Gold Test Verification):")
print(f"  Train Target: {sum(v['train'] for v in SPLIT_TARGETS.values()):,} (80%)")
print(f"  Val Target  : {sum(v['val']   for v in SPLIT_TARGETS.values()):,} (10% — 100% real scraped)")
print(f"  Test Target : {sum(v['test']  for v in SPLIT_TARGETS.values()):,} (10% — 100% real scraped + full manual verification)")
print(f"  Grand Total : {sum(v['train']+v['val']+v['test'] for v in SPLIT_TARGETS.values()):,}")


Day 3 Master Configuration Loaded (v3 — Full 3,000-Row Gold Test Verification):
  Train Target: 24,000 (80%)
  Val Target  : 3,000 (10% — 100% real scraped)
  Test Target : 3,000 (10% — 100% real scraped + full manual verification)
  Grand Total : 30,000


## Stage 1 (Both Members — Morning): Zero-Leakage Stratified 80/10/10 Split

### Rules Enforced Strictly in Stage 1:
1. **Text-Hash Deduplication First:** Ensures identical comments never cross splits.
2. **Quarantines External Train-Only Rows (`platform` starting with `twitter_dair`):** `0` DAIR-AI tweets enter `Val` or `Test`.
3. **Proportional Safeguard for Tight Classes (`Fear`):** When a scraped class pool is tight (`len(df_pool) < n_pick * 1.6`), Val+Test samples proportionally to that class's actual `(ENG, BN, BNG)` ratio so **zero languages are ever starved to 0 in Train**.
4. **Saves Locked Splits Immediately to Drive** (`01_locked_splits/`).


In [4]:
# ==============================================================================
# STAGE 1: ZERO-LEAKAGE STRATIFIED 80/10/10 SPLIT (WITH PROPORTIONAL SAFEGUARD)
# ==============================================================================
FORCE_RESPLIT = False  # Keep False to preserve locked splits; set True ONLY to rebuild from scratch

def norm_text_hash(txt: str) -> str:
    s = re.sub(r'\s+', ' ', str(txt).strip().lower())
    return hashlib.md5(s.encode('utf-8', errors='ignore')).hexdigest()

def stratified_pick_indices(df_pool: pd.DataFrame, n_pick: int, seed: int = 42) -> list:
    """
    Pick `n_pick` rows from `df_pool` (single class).
    - If pool has plenty of headroom (>= 1.6x n_pick): targets 40% ENG / 30% BN / 30% BNG & 70% Emoji.
    - If pool is tight (< 1.6x n_pick, e.g., scraped Fear=849 for 666 Val+Test): samples proportionally
      to the pool's actual language mix so NO language is ever starved to 0 in Train!
    Prefers higher ai_confidence (5 -> 4 -> 3) within each sub-bucket.
    """
    if len(df_pool) <= n_pick:
        return df_pool.index.tolist()

    if len(df_pool) < n_pick * 1.6:
        cell_weights = []
        for lg in ("ENG", "BN", "BNG"):
            lg_share = float((df_pool["lang"] == lg).mean())
            if lg_share > 0:
                lg_sub = df_pool[df_pool["lang"] == lg]
                emo_share = min(0.70, float(lg_sub["has_emoji"].mean()))
                cell_weights.append((lg, True,  lg_share * emo_share))
                cell_weights.append((lg, False, lg_share * (1.0 - emo_share)))
    else:
        cell_weights = [
            ("ENG", True,  0.28), ("ENG", False, 0.12),
            ("BN",  True,  0.21), ("BN",  False, 0.09),
            ("BNG", True,  0.21), ("BNG", False, 0.09),
        ]

    picked = []
    picked_set = set()
    rng = np.random.RandomState(seed)

    # Pass 1: Allocate up to target quota in each (lang, has_emoji) cell
    for lg, emo, w in cell_weights:
        sub = df_pool[(df_pool["lang"] == lg) & (df_pool["has_emoji"] == emo)]
        if sub.empty:
            continue
        target_cell = int(round(n_pick * w))
        take = min(len(sub), target_cell)
        if take > 0:
            sub_shuf = sub.iloc[rng.permutation(len(sub))]
            sub_pref = sub_shuf.sort_values("ai_confidence", ascending=False, kind="mergesort")
            chosen = sub_pref.index[:take].tolist()
            picked.extend(chosen)
            picked_set.update(chosen)

    if len(picked) > n_pick:
        picked = picked[:n_pick]
        picked_set = set(picked)

    # Pass 2: Top up any rounding remainder from remaining rows
    rem_need = n_pick - len(picked)
    if rem_need > 0:
        rem_df = df_pool.loc[~df_pool.index.isin(picked_set)]
        rem_shuf = rem_df.iloc[rng.permutation(len(rem_df))]
        rem_pref = rem_shuf.sort_values(["has_emoji", "ai_confidence"], ascending=[False, False], kind="mergesort")
        picked.extend(rem_pref.index[:rem_need].tolist())

    return picked

def run_stage1_locking_split():
    if (not FORCE_RESPLIT) and os.path.exists(CONFIG["VAL_LOCKED_PQ"]) and os.path.exists(CONFIG["TEST_LOCKED_PQ"]) and os.path.exists(CONFIG["TRAIN_PRE_AUG_PQ"]):
        print("[RESUME] Stage 1 locked splits already exist on Google Drive! Loading summary...")
        with open(CONFIG["SPLIT_REPORT_JSON"], "r", encoding="utf-8") as f:
            rep = json.load(f)
        print(f"  Val: {rep['val_total']:,} | Test: {rep['test_total']:,} | Pre-Aug Train: {rep['train_pre_aug_total']:,}")
        for c in CLASSES:
            ca = rep["per_class"][c]
            print(f"  {c:13s} | Train real={ca['train_pre_aug_real']:>4,} | AugNeeded={ca['aug_needed_in_stage3']:>4,} | Train langs={ca['train_lang_counts']}")
        return

    print("Loading Day 2 verified high-confidence dataset...")
    df = pd.read_parquet(DAY2_PARQUET) if os.path.exists(DAY2_PARQUET) else pd.read_csv(DAY2_CSV)
    initial_n = len(df)

    df = df[df["ai_label"].isin(CLASSES) & df["lang"].isin(["ENG", "BN", "BNG"])].copy()
    df["has_emoji"] = df["has_emoji"].astype(bool)
    df["ai_confidence"] = df["ai_confidence"].astype(int)
    df["text_hash"] = df["clean_text"].map(norm_text_hash)

    df = df.sort_values("ai_confidence", ascending=False).drop_duplicates(subset=["text_hash"], keep="first").reset_index(drop=True)
    print(f"Loaded {initial_n:,} rows -> {len(df):,} unique clean rows after text-hash deduplication.")

    df["is_external_train_only"] = df["platform"].astype(str).str.startswith("twitter_dair")
    df["is_augmented"] = False

    val_parts, test_parts, train_parts = [], [], []
    class_audit = {}

    for cls_idx, c in enumerate(CLASSES):
        sub_all     = df[df["ai_label"] == c]
        sub_scraped = sub_all[~sub_all["is_external_train_only"]]
        sub_ext     = sub_all[sub_all["is_external_train_only"]]

        n_val = SPLIT_TARGETS[c]["val"]
        n_test = SPLIT_TARGETS[c]["test"]
        n_train_target = SPLIT_TARGETS[c]["train"]

        assert len(sub_scraped) >= (n_val + n_test), (
            f"Class '{c}' only has {len(sub_scraped)} scraped rows, fewer than Val+Test ({n_val + n_test})!"
        )

        # 1. Lock Val + Test strictly from 100% REAL SCRAPED rows
        vt_idx = stratified_pick_indices(sub_scraped, n_val + n_test, seed=CONFIG["RANDOM_SEED"] + cls_idx * 10)
        df_vt  = sub_scraped.loc[vt_idx]

        val_idx  = stratified_pick_indices(df_vt, n_val, seed=CONFIG["RANDOM_SEED"] + cls_idx * 10 + 1)
        val_set  = set(val_idx)
        test_idx = [i for i in vt_idx if i not in val_set]

        df_val_c  = sub_scraped.loc[val_idx].copy()
        df_test_c = sub_scraped.loc[test_idx].copy()

        # 2. Remaining scraped rows + external Train-Only rows -> Pre-Aug Train Pool
        vt_set = set(vt_idx)
        rem_scraped_idx = [i for i in sub_scraped.index if i not in vt_set]
        df_train_cand = pd.concat([sub_scraped.loc[rem_scraped_idx], sub_ext], ignore_index=False)

        if len(df_train_cand) > n_train_target:
            tr_idx = stratified_pick_indices(df_train_cand, n_train_target, seed=CONFIG["RANDOM_SEED"] + cls_idx * 10 + 2)
            df_train_c = df_train_cand.loc[tr_idx].copy()
            downsampled = len(df_train_cand) - len(df_train_c)
            aug_needed = 0
        else:
            df_train_c = df_train_cand.copy()
            downsampled = 0
            aug_needed = n_train_target - len(df_train_c)

        val_parts.append(df_val_c)
        test_parts.append(df_test_c)
        train_parts.append(df_train_c)

        class_audit[c] = {
            "total_available": int(len(sub_all)),
            "scraped_available": int(len(sub_scraped)),
            "external_train_only": int(len(sub_ext)),
            "val_locked_real": int(len(df_val_c)),
            "test_locked_real": int(len(df_test_c)),
            "train_pre_aug_real": int(len(df_train_c)),
            "train_target": int(n_train_target),
            "downsampled_surplus": int(downsampled),
            "aug_needed_in_stage3": int(aug_needed),
            "train_lang_counts": df_train_c["lang"].value_counts().to_dict(),
            "val_lang_counts": df_val_c["lang"].value_counts().to_dict(),
            "test_lang_counts": df_test_c["lang"].value_counts().to_dict(),
        }

    df_val   = pd.concat(val_parts, ignore_index=True).sample(frac=1.0, random_state=CONFIG["RANDOM_SEED"]).reset_index(drop=True)
    df_test  = pd.concat(test_parts, ignore_index=True).sample(frac=1.0, random_state=CONFIG["RANDOM_SEED"]).reset_index(drop=True)
    df_train = pd.concat(train_parts, ignore_index=True).sample(frac=1.0, random_state=CONFIG["RANDOM_SEED"]).reset_index(drop=True)

    # Zero-leakage & split integrity assertions
    h_tr, h_va, h_te = set(df_train["text_hash"]), set(df_val["text_hash"]), set(df_test["text_hash"])
    assert len(h_tr & h_va) == 0, "LEAKAGE DETECTED: Train & Val overlap!"
    assert len(h_tr & h_te) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
    assert len(h_va & h_te) == 0, "LEAKAGE DETECTED: Val & Test overlap!"
    assert df_val["is_external_train_only"].sum() == 0, "External rows leaked into Val!"
    assert df_test["is_external_train_only"].sum() == 0, "External rows leaked into Test!"
    assert len(df_val) == 3_000 and len(df_test) == 3_000

    df_val.to_parquet(CONFIG["VAL_LOCKED_PQ"], index=False)
    df_test.to_parquet(CONFIG["TEST_LOCKED_PQ"], index=False)
    df_train.to_parquet(CONFIG["TRAIN_PRE_AUG_PQ"], index=False)

    report = {
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "input_unique_rows": int(len(df)),
        "val_total": int(len(df_val)),
        "test_total": int(len(df_test)),
        "train_pre_aug_total": int(len(df_train)),
        "total_aug_needed_stage3": int(sum(v["aug_needed_in_stage3"] for v in class_audit.values())),
        "zero_leakage_verified": True,
        "val_external_rows": 0,
        "test_external_rows": 0,
        "per_class": class_audit,
    }
    with open(CONFIG["SPLIT_REPORT_JSON"], "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    print("\n" + "=" * 105)
    print(f"STAGE 1 COMPLETE (ZERO LEAKAGE VERIFIED) | Val: {len(df_val):,} | Test: {len(df_test):,} | Pre-Aug Train: {len(df_train):,}")
    print("=" * 105)
    print(f"{'Class':13s} | {'Val(Real)':>9s} | {'Test(Real)':>10s} | {'Train(Real)':>11s} | {'TrainTarget':>11s} | {'AugNeeded':>9s} | {'Train ENG/BN/BNG'}")
    print("-" * 105)
    for c in CLASSES:
        ca = class_audit[c]
        lc = ca["train_lang_counts"]
        lstr = f"{lc.get('ENG',0):>4}/{lc.get('BN',0):>4}/{lc.get('BNG',0):>4}"
        print(f"{c:13s} | {ca['val_locked_real']:>9,} | {ca['test_locked_real']:>10,} | {ca['train_pre_aug_real']:>11,} | {ca['train_target']:>11,} | {ca['aug_needed_in_stage3']:>9,} | {lstr}")
    print("=" * 105)
    print("Saved Stage 1 locked splits to:", DIRS['splits'])

    del df, df_val, df_test, df_train
    gc.collect()

run_stage1_locking_split()


Loading Day 2 verified high-confidence dataset...
Loaded 44,580 rows -> 44,580 unique clean rows after text-hash deduplication.

STAGE 1 COMPLETE (ZERO LEAKAGE VERIFIED) | Val: 3,000 | Test: 3,000 | Pre-Aug Train: 20,472
Class         | Val(Real) | Test(Real) | Train(Real) | TrainTarget | AugNeeded | Train ENG/BN/BNG
---------------------------------------------------------------------------------------------------------
Joy           |       334 |        334 |       2,666 |       2,666 |         0 | 1076/ 837/ 753
Neutral       |       334 |        334 |       2,666 |       2,666 |         0 |  962/1129/ 575
Anger         |       333 |        333 |       2,054 |       2,667 |       613 |  486/1176/ 392
Sadness       |       334 |        334 |       2,666 |       2,666 |         0 | 1167/ 881/ 618
Surprise      |       333 |        333 |       1,626 |       2,667 |     1,041 | 1018/ 383/ 225
Sarcasm       |       333 |        333 |       2,667 |       2,667 |         0 | 1054/ 893/ 720

## Stage 2 (Both Members): Full 3,000-Row Gold Test Verification & Cohen’s Kappa

*(No AI is used here — this is manual human verification of 100% of your locked `test_3k` split to compute Human vs. AI Cohen’s Kappa and upgrade `test_3k` to a 100% Human-Verified Gold Test Set!)*

1. **Run Stage 2A** → Splits all **3,000 locked Test rows** evenly into two blinded, styled Excel workbooks (`1,500` rows each, balanced `167`/`166` per class) with built-in 9-class dropdown validation in `MyDrive/AffectFusion_Day3/02_gold_verification/`:
   - **`gold_member1_ragib_1500rows.xlsx`**
   - **`gold_member2_galib_1500rows.xlsx`**
2. **Verify manually in parallel** (fill the yellow `human_label` column and save back to Drive). You can immediately run **Stage 3A → Stage 3B → Stage 4** in parallel while annotating!
3. **Run Stage 2B** anytime (works incrementally on whatever rows you've filled so far, e.g., at 500, 1,500, or all 3,000 rows) to compute **Cohen’s Kappa ($\kappa$)**, raw agreement %, per-language agreement, and your 9×9 Human-vs-AI Confusion Matrix.
   - Once all **3,000 rows** are filled and `APPLY_GOLD_LABELS_TO_TEST = True`, Stage 2B automatically upgrades `test_3k_locked.parquet` with your human-verified ground truth (saving original AI labels in `ai_label_pre_gold`). Simply re-run **Stage 4** afterward so `test_3k.csv` and `test_3k.parquet` contain your 100% Gold Test labels!


In [5]:
# ==============================================================================
# STAGE 2A: EXPORT BLINDED 3,000-ROW GOLD TEST WORKBOOKS (1,500 ROWS EACH)
# ==============================================================================
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation

def style_and_write_gold_xlsx(df_member: pd.DataFrame, out_path: str, member_title: str):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Manual_Annotation"

    ws_ref = wb.create_sheet("Rubric_Guide")
    rubric_rows = [
        ("Emotion Class", "Quick Annotation Definition"),
        ("Joy",          "Genuine happiness, praise, love, gratitude, pride, celebration, genuine laughter"),
        ("Neutral",      "Objective facts, plain questions, timestamps, announcements, flat statements"),
        ("Anger",        "Outrage, hostility, strong complaints, demanding punishment/boycott, cursing"),
        ("Sadness",      "Grief, heartbreak, missing someone/pet, sympathy, condolences, regret"),
        ("Surprise",     "Genuine shock, astonishment, disbelief, unexpected plot twist or realization"),
        ("Sarcasm",      "Irony, mockery, passive-aggressive praise, or ironic emoji-text contradiction"),
        ("Fear",         "Genuine dread, terror, anxiety, horror, worry, warning of danger/disaster"),
        ("Disgust",      "Revulsion, calling something gross/cringe/corrupt/scam/shameless/faltu/ghrina"),
        ("Anticipation", "Looking forward to something, hype, waiting eagerly, requesting next part/episode"),
    ]
    for r_i, (k, v) in enumerate(rubric_rows, start=1):
        ws_ref.cell(row=r_i, column=1, value=k)
        ws_ref.cell(row=r_i, column=2, value=v)
        if r_i == 1:
            for c_i in (1, 2):
                ws_ref.cell(row=r_i, column=c_i).font = Font(bold=True, color="FFFFFF")
                ws_ref.cell(row=r_i, column=c_i).fill = PatternFill("solid", fgColor="1F2A44")
    ws_ref.column_dimensions["A"].width = 18
    ws_ref.column_dimensions["B"].width = 88

    headers = ["sample_no", "row_id", "lang", "has_emoji", "clean_text", "emojis", "human_label", "notes_optional"]
    header_fill = PatternFill("solid", fgColor="1F2A44")
    header_font = Font(bold=True, color="FFFFFF", size=11)
    input_fill  = PatternFill("solid", fgColor="FFF9E6")
    zebra_fill  = PatternFill("solid", fgColor="F5F7FA")
    thin_border = Border(
        left=Side(style="thin", color="D0D7DE"), right=Side(style="thin", color="D0D7DE"),
        top=Side(style="thin", color="D0D7DE"), bottom=Side(style="thin", color="D0D7DE")
    )

    ws.append(headers)
    for c_i in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=c_i)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = thin_border
    ws.row_dimensions[1].height = 24

    for idx, row in enumerate(df_member.itertuples(index=False), start=1):
        r_num = idx + 1
        vals = [idx, row.row_id, row.lang, "Yes" if row.has_emoji else "No", row.clean_text, row.emojis, "", ""]
        ws.append(vals)
        for c_i in range(1, len(headers) + 1):
            cell = ws.cell(row=r_num, column=c_i)
            cell.border = thin_border
            if c_i == 7:
                cell.fill = input_fill
                cell.alignment = Alignment(horizontal="center", vertical="center")
            elif idx % 2 == 0:
                cell.fill = zebra_fill
            if c_i == 5:
                cell.alignment = Alignment(wrap_text=True, vertical="top")
            elif c_i in (1, 2, 3, 4, 6):
                cell.alignment = Alignment(horizontal="center", vertical="center")

    dv = DataValidation(type="list", formula1="=Rubric_Guide!$A$2:$A$10", allow_blank=True)
    dv.error = "Select one of the 9 valid emotion classes from the dropdown."
    dv.errorTitle = "Invalid Emotion Class"
    ws.add_data_validation(dv)
    dv.add(f"G2:G{len(df_member) + 1}")

    for col_l, w in {"A": 11, "B": 14, "C": 8, "D": 11, "E": 72, "F": 14, "G": 18, "H": 24}.items():
        ws.column_dimensions[col_l].width = w
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:H{len(df_member) + 1}"

    wb.save(out_path)
    print(f"  [SAVED] {member_title}: {out_path} ({len(df_member):,} rows)")

def run_stage2a_export_gold_sheets():
    if os.path.exists(CONFIG["GOLD_M1_XLSX"]) and os.path.exists(CONFIG["GOLD_M2_XLSX"]) and os.path.exists(CONFIG["GOLD_KEY_PQ"]):
        print("[RESUME] Gold 3,000 verification Excel sheets already exist on Google Drive:")
        print("  -", CONFIG["GOLD_M1_XLSX"])
        print("  -", CONFIG["GOLD_M2_XLSX"])
        return

    df_test = pd.read_parquet(CONFIG["TEST_LOCKED_PQ"])
    # Use pre-gold AI label if test_3k was already upgraded earlier
    label_col = "ai_label_pre_gold" if "ai_label_pre_gold" in df_test.columns else "ai_label"
    m1_frames, m2_frames, key_frames = [], [], []
    odd_toggle = 0

    for cls_idx, c in enumerate(CLASSES):
        sub = df_test[df_test[label_col] == c].sample(
            frac=1.0, random_state=CONFIG["RANDOM_SEED"] + cls_idx
        ).reset_index(drop=True)

        n = len(sub)
        if n % 2 == 0:
            split_at = n // 2
        else:
            split_at = (n // 2) + (1 if odd_toggle % 2 == 0 else 0)
            odd_toggle += 1

        m1 = sub.iloc[:split_at].copy()
        m2 = sub.iloc[split_at:].copy()
        m1["assigned_annotator"] = "Member1_Ragib"
        m2["assigned_annotator"] = "Member2_Galib"

        m1_frames.append(m1)
        m2_frames.append(m2)
        key_frames.append(pd.concat([m1, m2], ignore_index=True))

    df_m1  = pd.concat(m1_frames, ignore_index=True).sample(frac=1.0, random_state=CONFIG["RANDOM_SEED"] + 1).reset_index(drop=True)
    df_m2  = pd.concat(m2_frames, ignore_index=True).sample(frac=1.0, random_state=CONFIG["RANDOM_SEED"] + 2).reset_index(drop=True)
    df_key = pd.concat(key_frames, ignore_index=True)

    assert len(df_m1) == 1500 and len(df_m2) == 1500 and len(df_key) == 3000, (
        f"Unexpected counts: M1={len(df_m1)}, M2={len(df_m2)}, Key={len(df_key)}"
    )

    df_key.to_parquet(CONFIG["GOLD_KEY_PQ"], index=False)
    style_and_write_gold_xlsx(df_m1, CONFIG["GOLD_M1_XLSX"], "Member 1 (Ragib — 1,500 rows)")
    style_and_write_gold_xlsx(df_m2, CONFIG["GOLD_M2_XLSX"], "Member 2 (Galib — 1,500 rows)")

    print("\nGold 3,000 Verification Set Ready (1,500 rows each)! Fill `human_label` in both Excel sheets, then run Stage 2B.")
    del df_test, df_m1, df_m2, df_key
    gc.collect()

run_stage2a_export_gold_sheets()


  [SAVED] Member 1 (Ragib — 1,500 rows): /content/drive/MyDrive/AffectFusion_Day3/02_gold_verification/gold_member1_ragib_1500rows.xlsx (1,500 rows)
  [SAVED] Member 2 (Galib — 1,500 rows): /content/drive/MyDrive/AffectFusion_Day3/02_gold_verification/gold_member2_galib_1500rows.xlsx (1,500 rows)

Gold 3,000 Verification Set Ready (1,500 rows each)! Fill `human_label` in both Excel sheets, then run Stage 2B.


In [6]:
# ==============================================================================
# STAGE 2B: COMPUTE COHEN'S KAPPA (HUMAN vs. AI) & OPTIONAL GOLD TEST UPGRADE
# ==============================================================================
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix

# Set True to upgrade test_3k_locked.parquet with 100% human-verified ground truth
# automatically once all 3,000 rows are completed (preserves original AI labels in `ai_label_pre_gold`)!
APPLY_GOLD_LABELS_TO_TEST = True

def run_stage2b_compute_cohens_kappa():
    df_key = pd.read_parquet(CONFIG["GOLD_KEY_PQ"])
    ai_col = "ai_label_pre_gold" if "ai_label_pre_gold" in df_key.columns else "ai_label"
    df_key = df_key[["row_id", "lang", "has_emoji", "clean_text", ai_col, "ai_confidence", "assigned_annotator"]].rename(columns={ai_col: "ai_label"})

    df_m1  = pd.read_excel(CONFIG["GOLD_M1_XLSX"], sheet_name="Manual_Annotation")
    df_m2  = pd.read_excel(CONFIG["GOLD_M2_XLSX"], sheet_name="Manual_Annotation")
    df_hum = pd.concat([df_m1[["row_id", "human_label"]], df_m2[["row_id", "human_label"]]], ignore_index=True)

    df_hum["human_label"] = df_hum["human_label"].astype(str).str.strip()
    df_hum = df_hum[df_hum["human_label"].isin(CLASSES)].drop_duplicates(subset=["row_id"], keep="last").copy()

    if len(df_hum) == 0:
        print("[WAITING] No filled `human_label` rows found yet in the Excel files.")
        print("  Fill `human_label` in both Drive Excel files, then re-run this cell!")
        print("  (Proceed directly to Stage 3A -> 3B -> 4 right now while annotating in parallel.)")
        return

    merged = df_key.merge(df_hum, on="row_id", how="inner")
    y_ai, y_hum = merged["ai_label"].tolist(), merged["human_label"].tolist()

    kappa   = float(cohen_kappa_score(y_hum, y_ai, labels=CLASSES))
    raw_acc = float(accuracy_score(y_hum, y_ai))
    interp  = (
        "Almost Perfect Agreement" if kappa >= 0.81 else
        "Substantial Agreement"    if kappa >= 0.61 else
        "Moderate Agreement"       if kappa >= 0.41 else
        "Fair / Needs Review"
    )

    lang_metrics = {}
    for lg in ("ENG", "BN", "BNG"):
        sub = merged[merged["lang"] == lg]
        if len(sub) >= 10:
            lang_metrics[lg] = {
                "n": int(len(sub)),
                "agreement_pct": round(100.0 * accuracy_score(sub["human_label"], sub["ai_label"]), 2),
                "cohens_kappa": round(float(cohen_kappa_score(sub["human_label"], sub["ai_label"], labels=CLASSES)), 4),
            }

    annotator_counts = merged["assigned_annotator"].value_counts().to_dict()
    per_cls_acc = {
        c: {"n_ai": int(len(sub)), "human_match_pct": round(100.0 * (sub["human_label"] == c).mean(), 2)}
        for c in CLASSES for sub in [merged[merged["ai_label"] == c]] if len(sub) > 0
    }

    cm_df = pd.DataFrame(
        confusion_matrix(y_hum, y_ai, labels=CLASSES),
        index=[f"Human_{c}" for c in CLASSES],
        columns=[f"AI_{c}" for c in CLASSES]
    )

    merged.to_csv(CONFIG["GOLD_MERGED_CSV"], index=False, encoding="utf-8-sig")
    report = {
        "evaluated_at": datetime.now(timezone.utc).isoformat(),
        "verified_rows": int(len(merged)),
        "target_gold_rows": 3000,
        "annotator_progress": annotator_counts,
        "raw_agreement_pct": round(100.0 * raw_acc, 2),
        "cohens_kappa": round(kappa, 4),
        "landis_koch_interpretation": interp,
        "by_language": lang_metrics,
        "ai_precision_by_class": per_cls_acc,
    }
    with open(CONFIG["KAPPA_REPORT_JSON"], "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    print("\n" + "=" * 78)
    print(f"GOLD SET EVALUATION ({len(merged):,} / 3,000 verified rows)")
    print("=" * 78)
    print(f"  Progress      : {annotator_counts}")
    print(f"  Raw Agreement : {100.0 * raw_acc:.2f}%")
    print(f"  Cohen's Kappa : {kappa:.4f}  ({interp})")
    print("  By Language   :", json.dumps(lang_metrics, indent=2))
    print("\nHuman (rows) vs. AI (cols) Confusion Matrix:")
    display(cm_df)
    print(f"\nSaved Gold Report to Drive: {CONFIG['KAPPA_REPORT_JSON']}")

    # Upgrade locked test_3k with 100% human-verified ground truth once all 3,000 rows are done!
    if APPLY_GOLD_LABELS_TO_TEST and len(merged) == len(df_key):
        df_test_full = pd.read_parquet(CONFIG["TEST_LOCKED_PQ"])
        gold_map = dict(zip(merged["row_id"], merged["human_label"]))
        if "ai_label_pre_gold" not in df_test_full.columns:
            df_test_full["ai_label_pre_gold"] = df_test_full["ai_label"]
        df_test_full["ai_label"] = df_test_full["row_id"].map(gold_map).fillna(df_test_full["ai_label"])
        df_test_full.to_parquet(CONFIG["TEST_LOCKED_PQ"], index=False)
        n_changed = int((df_test_full["ai_label"] != df_test_full["ai_label_pre_gold"]).sum())
        print(f"\n  [GOLD UPGRADE COMPLETE] Updated test_3k_locked.parquet with 3,000 human-verified labels ({n_changed:,} corrected)!")
        print("  Updated Gold Test Class Counts:", df_test_full["ai_label"].value_counts().to_dict())
        print("  -> Re-run Stage 4 now so test_3k.csv & test_3k.parquet reflect your 100% Gold Test labels!")
    elif APPLY_GOLD_LABELS_TO_TEST and len(merged) < len(df_key):
        print(f"\n  [INFO] {len(df_key) - len(merged):,} rows still pending before automatic Gold Test upgrade triggers.")

run_stage2b_compute_cohens_kappa()



GOLD SET EVALUATION (3,000 / 3,000 verified rows)
  Progress      : {'Member1_Ragib': 1500, 'Member2_Galib': 1500}
  Raw Agreement : 80.87%
  Cohen's Kappa : 0.7847  (Substantial Agreement)
  By Language   : {
  "ENG": {
    "n": 1147,
    "agreement_pct": 80.91,
    "cohens_kappa": 0.7846
  },
  "BN": {
    "n": 1033,
    "agreement_pct": 81.32,
    "cohens_kappa": 0.7869
  },
  "BNG": {
    "n": 820,
    "agreement_pct": 80.24,
    "cohens_kappa": 0.7762
  }
}

Human (rows) vs. AI (cols) Confusion Matrix:


,AI_Joy,AI_Neutral,AI_Anger,AI_Sadness,AI_Surprise,AI_Sarcasm,AI_Fear,AI_Disgust,AI_Anticipation
Human_Joy,312,46,5,3,25,17,2,4,22
Human_Neutral,9,266,3,5,35,5,11,4,8
Human_Anger,1,0,276,5,7,19,5,55,2
Human_Sadness,3,2,6,293,7,3,4,11,4
Human_Surprise,0,2,3,2,199,1,2,0,1
Human_Sarcasm,5,15,18,9,29,270,4,37,6
Human_Fear,0,0,1,13,24,0,303,3,1
Human_Disgust,0,2,19,4,5,18,2,218,0
Human_Anticipation,4,1,2,0,2,0,0,1,289



Saved Gold Report to Drive: /content/drive/MyDrive/AffectFusion_Day3/logs/stage2_cohens_kappa_3000_report.json

  [GOLD UPGRADE COMPLETE] Updated test_3k_locked.parquet with 3,000 human-verified labels (574 corrected)!
  Updated Gold Test Class Counts: {'Joy': 436, 'Sarcasm': 393, 'Anger': 370, 'Neutral': 346, 'Fear': 345, 'Sadness': 333, 'Anticipation': 299, 'Disgust': 268, 'Surprise': 210}
  -> Re-run Stage 4 now so test_3k.csv & test_3k.parquet reflect your 100% Gold Test labels!


In [7]:
# ==============================================================================
# STAGE 2C: EXPORT ADDITIONAL 2,000-ROW GOLD VERIFICATION BATCH (1,000 EACH)
# ==============================================================================
# Keeps your verified 3,000-row Test workbooks 100% untouched!
# Recommended: SOURCE_SPLIT = "val" (100% real scraped rows, upgrades 66.7% of val_3k,
#              and does NOT require re-running Stage 3 augmentation!)
# Optional:    SOURCE_SPLIT = "train" (samples 2,000 real rows from train_pre_aug)
SOURCE_SPLIT = "val"   # "val" (recommended) or "train"

EXTRA_TOTAL_ROWS = 2000
CONFIG.update({
    "GOLD2_SOURCE_SPLIT": SOURCE_SPLIT,
    "GOLD2_M1_XLSX":      os.path.join(DIRS['gold'], f"gold2_{SOURCE_SPLIT}_member1_ragib_1000rows.xlsx"),
    "GOLD2_M2_XLSX":      os.path.join(DIRS['gold'], f"gold2_{SOURCE_SPLIT}_member2_galib_1000rows.xlsx"),
    "GOLD2_KEY_PQ":       os.path.join(DIRS['gold'], f"gold2_{SOURCE_SPLIT}_2000_ai_key_internal.parquet"),
    "GOLD2_MERGED_CSV":   os.path.join(DIRS['gold'], f"gold2_{SOURCE_SPLIT}_2000_verified_merged.csv"),
    "KAPPA_5K_REPORT_JS": os.path.join(DIRS['logs'], f"stage2_cohens_kappa_combined_5000_report.json"),
})

def run_stage2c_export_extra_2k_sheets():
    if os.path.exists(CONFIG["GOLD2_M1_XLSX"]) and os.path.exists(CONFIG["GOLD2_M2_XLSX"]) and os.path.exists(CONFIG["GOLD2_KEY_PQ"]):
        print("[RESUME] Additional 2,000-row Gold Excel sheets already exist on Google Drive:")
        print("  -", CONFIG["GOLD2_M1_XLSX"])
        print("  -", CONFIG["GOLD2_M2_XLSX"])
        return

    src_pq = CONFIG["VAL_LOCKED_PQ"] if SOURCE_SPLIT == "val" else CONFIG["TRAIN_PRE_AUG_PQ"]
    df_src = pd.read_parquet(src_pq)
    label_col = "ai_label_pre_gold" if "ai_label_pre_gold" in df_src.columns else "ai_label"

    # Exclude any row_ids already in the 3,000-row Gold key (zero overlap guarantee)
    used_ids = set()
    if os.path.exists(CONFIG["GOLD_KEY_PQ"]):
        used_ids.update(pd.read_parquet(CONFIG["GOLD_KEY_PQ"], columns=["row_id"])["row_id"].tolist())
    df_src = df_src[~df_src["row_id"].isin(used_ids)].copy()

    # Exact per-class targets summing to 2,000 (223 for Joy & Neutral; 222 for the other 7)
    base_n, rem_n = divmod(EXTRA_TOTAL_ROWS, len(CLASSES))
    m1_frames, m2_frames, key_frames = [], [], []
    odd_toggle = 0

    for cls_idx, c in enumerate(CLASSES):
        n_cls = base_n + (1 if cls_idx < rem_n else 0)
        sub = df_src[df_src[label_col] == c]
        assert len(sub) >= n_cls, f"Class '{c}' in {SOURCE_SPLIT} only has {len(sub)} rows (< {n_cls})!"

        pick_idx = stratified_pick_indices(sub, n_cls, seed=CONFIG["RANDOM_SEED"] + 2000 + cls_idx)
        sub_pick = sub.loc[pick_idx].sample(frac=1.0, random_state=CONFIG["RANDOM_SEED"] + 2100 + cls_idx).reset_index(drop=True)

        if n_cls % 2 == 0:
            split_at = n_cls // 2
        else:
            split_at = (n_cls // 2) + (1 if odd_toggle % 2 == 0 else 0)
            odd_toggle += 1

        m1 = sub_pick.iloc[:split_at].copy()
        m2 = sub_pick.iloc[split_at:].copy()
        m1["assigned_annotator"] = "Member1_Ragib"
        m2["assigned_annotator"] = "Member2_Galib"

        m1_frames.append(m1)
        m2_frames.append(m2)
        key_frames.append(pd.concat([m1, m2], ignore_index=True))

    df_m1  = pd.concat(m1_frames, ignore_index=True).sample(frac=1.0, random_state=CONFIG["RANDOM_SEED"] + 11).reset_index(drop=True)
    df_m2  = pd.concat(m2_frames, ignore_index=True).sample(frac=1.0, random_state=CONFIG["RANDOM_SEED"] + 12).reset_index(drop=True)
    df_key = pd.concat(key_frames, ignore_index=True)

    assert len(df_m1) == 1000 and len(df_m2) == 1000 and len(df_key) == 2000

    df_key.to_parquet(CONFIG["GOLD2_KEY_PQ"], index=False)
    style_and_write_gold_xlsx(df_m1, CONFIG["GOLD2_M1_XLSX"], f"Member 1 (Ragib — 1,000 {SOURCE_SPLIT.upper()} rows)")
    style_and_write_gold_xlsx(df_m2, CONFIG["GOLD2_M2_XLSX"], f"Member 2 (Galib — 1,000 {SOURCE_SPLIT.upper()} rows)")

    print(f"\nBatch 2 ({SOURCE_SPLIT.upper()} — 2,000 rows; 1,000 each) exported to Drive! Fill `human_label`, then run Stage 2D.")
    del df_src, df_m1, df_m2, df_key
    gc.collect()

run_stage2c_export_extra_2k_sheets()

  [SAVED] Member 1 (Ragib — 1,000 VAL rows): /content/drive/MyDrive/AffectFusion_Day3/02_gold_verification/gold2_val_member1_ragib_1000rows.xlsx (1,000 rows)
  [SAVED] Member 2 (Galib — 1,000 VAL rows): /content/drive/MyDrive/AffectFusion_Day3/02_gold_verification/gold2_val_member2_galib_1000rows.xlsx (1,000 rows)

Batch 2 (VAL — 2,000 rows; 1,000 each) exported to Drive! Fill `human_label`, then run Stage 2D.


In [8]:
# ==============================================================================
# STAGE 2D: SCORE BATCH 2 (2K) + COMBINED 5,000-ROW COHEN'S KAPPA & GOLD UPGRADE
# ==============================================================================
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix

APPLY_GOLD2_LABELS_TO_SPLIT = True   # Upgrades val_3k_locked.parquet (or train_pre_aug.parquet) once all 2,000 are done

def run_stage2d_score_2k_and_combined_5k():
    df_key2 = pd.read_parquet(CONFIG["GOLD2_KEY_PQ"])
    ai_col2 = "ai_label_pre_gold" if "ai_label_pre_gold" in df_key2.columns else "ai_label"
    df_key2 = df_key2[["row_id", "lang", "has_emoji", "clean_text", ai_col2, "ai_confidence", "assigned_annotator"]].rename(columns={ai_col2: "ai_label"})

    df_m1 = pd.read_excel(CONFIG["GOLD2_M1_XLSX"], sheet_name="Manual_Annotation")
    df_m2 = pd.read_excel(CONFIG["GOLD2_M2_XLSX"], sheet_name="Manual_Annotation")
    df_hum2 = pd.concat([df_m1[["row_id", "human_label"]], df_m2[["row_id", "human_label"]]], ignore_index=True)
    df_hum2["human_label"] = df_hum2["human_label"].astype(str).str.strip()
    df_hum2 = df_hum2[df_hum2["human_label"].isin(CLASSES)].drop_duplicates(subset=["row_id"], keep="last").copy()

    if len(df_hum2) == 0:
        print("[WAITING] No filled `human_label` rows found yet in the Batch 2 (2K) Excel files.")
        return

    merged2 = df_key2.merge(df_hum2, on="row_id", how="inner")
    merged2["batch"] = f"batch2_{CONFIG['GOLD2_SOURCE_SPLIT']}_2k"
    merged2.to_csv(CONFIG["GOLD2_MERGED_CSV"], index=False, encoding="utf-8-sig")

    k2   = float(cohen_kappa_score(merged2["human_label"], merged2["ai_label"], labels=CLASSES))
    acc2 = float(accuracy_score(merged2["human_label"], merged2["ai_label"]))
    print("=" * 78)
    print(f"BATCH 2 ({CONFIG['GOLD2_SOURCE_SPLIT'].upper()}) EVALUATION: {len(merged2):,} / 2,000 verified")
    print(f"  Raw Agreement : {100.0 * acc2:.2f}%  |  Cohen's Kappa : {k2:.4f}")
    print("=" * 78)

    combined_report = {
        "evaluated_at": datetime.now(timezone.utc).isoformat(),
        "batch2_split": CONFIG["GOLD2_SOURCE_SPLIT"],
        "batch2_verified_rows": int(len(merged2)),
        "batch2_raw_agreement_pct": round(100.0 * acc2, 2),
        "batch2_cohens_kappa": round(k2, 4),
    }

    # Combine with Batch 1 (3,000 Test rows) -> 5,000-Row Overall Kappa!
    if os.path.exists(CONFIG["GOLD_MERGED_CSV"]):
        merged1 = pd.read_csv(CONFIG["GOLD_MERGED_CSV"])
        merged1["batch"] = "batch1_test_3k"
        common_cols = ["row_id", "lang", "has_emoji", "clean_text", "ai_label", "human_label", "assigned_annotator", "batch"]
        df_5k = pd.concat([merged1[common_cols], merged2[common_cols]], ignore_index=True).drop_duplicates(subset=["row_id"], keep="last")

        k_all   = float(cohen_kappa_score(df_5k["human_label"], df_5k["ai_label"], labels=CLASSES))
        acc_all = float(accuracy_score(df_5k["human_label"], df_5k["ai_label"]))
        interp  = (
            "Almost Perfect Agreement" if k_all >= 0.81 else
            "Substantial Agreement"    if k_all >= 0.61 else
            "Moderate Agreement"       if k_all >= 0.41 else
            "Fair / Needs Review"
        )
        lang_5k = {
            lg: {
                "n": int(len(sub)),
                "agreement_pct": round(100.0 * accuracy_score(sub["human_label"], sub["ai_label"]), 2),
                "cohens_kappa": round(float(cohen_kappa_score(sub["human_label"], sub["ai_label"], labels=CLASSES)), 4),
            }
            for lg in ("ENG", "BN", "BNG") for sub in [df_5k[df_5k["lang"] == lg]] if len(sub) >= 10
        }

        combined_5k_csv = os.path.join(DIRS["gold"], "gold_5000_combined_verified.csv")
        df_5k.to_csv(combined_5k_csv, index=False, encoding="utf-8-sig")

        combined_report.update({
            "combined_verified_rows": int(len(df_5k)),
            "combined_raw_agreement_pct": round(100.0 * acc_all, 2),
            "combined_cohens_kappa": round(k_all, 4),
            "landis_koch_interpretation": interp,
            "combined_by_language": lang_5k,
        })

        print(f"\nCOMBINED GOLD EVALUATION ({len(df_5k):,} / 5,000 total verified rows)")
        print(f"  Raw Agreement : {100.0 * acc_all:.2f}%")
        print(f"  Cohen's Kappa : {k_all:.4f}  ({interp})")
        print("  By Language   :", json.dumps(lang_5k, indent=2))
        cm_df = pd.DataFrame(
            confusion_matrix(df_5k["human_label"], df_5k["ai_label"], labels=CLASSES),
            index=[f"Human_{c}" for c in CLASSES],
            columns=[f"AI_{c}" for c in CLASSES]
        )
        display(cm_df)

    with open(CONFIG["KAPPA_5K_REPORT_JS"], "w", encoding="utf-8") as f:
        json.dump(combined_report, f, indent=2)
    print(f"\nSaved combined report to Drive: {CONFIG['KAPPA_5K_REPORT_JS']}")

    # Upgrade val_3k_locked.parquet (or train_pre_aug.parquet) once all 2,000 rows are done
    if APPLY_GOLD2_LABELS_TO_SPLIT and len(merged2) == len(df_key2):
        target_pq = CONFIG["VAL_LOCKED_PQ"] if CONFIG["GOLD2_SOURCE_SPLIT"] == "val" else CONFIG["TRAIN_PRE_AUG_PQ"]
        df_tgt = pd.read_parquet(target_pq)
        gold2_map = dict(zip(merged2["row_id"], merged2["human_label"]))
        if "ai_label_pre_gold" not in df_tgt.columns:
            df_tgt["ai_label_pre_gold"] = df_tgt["ai_label"]
        df_tgt["ai_label"] = df_tgt["row_id"].map(gold2_map).fillna(df_tgt["ai_label"])
        df_tgt.to_parquet(target_pq, index=False)
        n_chg = int((df_tgt["ai_label"] != df_tgt["ai_label_pre_gold"]).sum())
        print(f"\n  [GOLD UPGRADE] Updated {os.path.basename(target_pq)} with 2,000 human-verified labels ({n_chg:,} corrected)!")
        if CONFIG["GOLD2_SOURCE_SPLIT"] == "val":
            print("  -> Re-run Stage 4 so val_3k.csv & val_3k.parquet reflect your verified labels (Stage 3 does NOT need re-running).")
        else:
            print("  -> Since train_pre_aug changed, delete 03_train_augmented/* and re-run Stage 3B -> Stage 4.")

run_stage2d_score_2k_and_combined_5k()

BATCH 2 (VAL) EVALUATION: 2,000 / 2,000 verified
  Raw Agreement : 86.20%  |  Cohen's Kappa : 0.8447

COMBINED GOLD EVALUATION (5,000 / 5,000 total verified rows)
  Raw Agreement : 83.00%
  Cohen's Kappa : 0.8087  (Substantial Agreement)
  By Language   : {
  "ENG": {
    "n": 1933,
    "agreement_pct": 83.39,
    "cohens_kappa": 0.8129
  },
  "BN": {
    "n": 1653,
    "agreement_pct": 83.48,
    "cohens_kappa": 0.8128
  },
  "BNG": {
    "n": 1414,
    "agreement_pct": 81.9,
    "cohens_kappa": 0.7958
  }
}


,AI_Joy,AI_Neutral,AI_Anger,AI_Sadness,AI_Surprise,AI_Sarcasm,AI_Fear,AI_Disgust,AI_Anticipation
Human_Joy,526,75,5,4,33,24,2,7,36
Human_Neutral,12,449,4,6,42,5,13,8,15
Human_Anger,1,3,468,11,9,35,5,98,4
Human_Sadness,3,3,8,499,8,3,6,15,5
Human_Surprise,0,5,3,2,377,3,3,0,1
Human_Sarcasm,8,17,23,12,40,456,10,46,7
Human_Fear,1,1,1,16,36,0,513,3,1
Human_Disgust,1,3,40,6,6,29,3,376,0
Human_Anticipation,5,1,3,0,4,0,0,2,486



Saved combined report to Drive: /content/drive/MyDrive/AffectFusion_Day3/logs/stage2_cohens_kappa_combined_5000_report.json

  [GOLD UPGRADE] Updated val_3k_locked.parquet with 2,000 human-verified labels (276 corrected)!
  -> Re-run Stage 4 so val_3k.csv & val_3k.parquet reflect your verified labels (Stage 3 does NOT need re-running).


## Stage 3A (Member 1): Gemini Flash Paraphraser Setup (Only ~143 Calls for All 3,528 Aug Rows!)

### How Gemini Flash Is Used in Day 3:
- **NOT for labeling** (all 44,580 rows are already labeled).
- **ONLY for high-quality, emoji-preserving Train-Only Paraphrasing** on the 5 deficit classes (`Fear` +1,384, `Surprise` +1,041, `Anger` +613, `Disgust` +396, `Anticipation` +94 = `3,528` rows → **only ~143 Gemini API calls = ~6–11 minutes on CPU**!).
- **Smart Cross-Lingual Support:** For `Fear`, Gemini Flash adapts real English `Fear` Train seeds into fluent Bangladeshi `BN` and `BNG` (avoiding repetitive paraphrasing of small seed pools) while keeping exact emotion semantics and original/natural emojis.
- **Automatic Zero-API Fallback:** If you set `USE_GEMINI_AUGMENTATION = False`, provide no key, or hit daily quota mid-run, Stage 3B seamlessly finishes remaining rows locally via Emotion-Safe Lexical Paraphrasing (~10 seconds, zero crash!).


In [12]:
# ==============================================================================
# STAGE 3A: GEMINI FLASH MULTI-KEY x MULTI-MODEL PARAPHRASING POOL
# ==============================================================================
from typing import List
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# Set True to use Gemini Flash for natural ENG/BN/BNG paraphrasing (~143 calls total)
# Set False for instant 10-second 100% Zero-API local lexical paraphrasing
USE_GEMINI_AUGMENTATION = True

# Option 1 (Recommended): Store keys in Colab Secrets as GEMINI_API_KEY, GEMINI_API_KEY_2, etc.
# Option 2: Paste directly into MANUAL_GEMINI_API_KEYS below (remove the #)
MANUAL_GEMINI_API_KEYS = [
    "YOUR_GEMINI_API_KEY_HERE",
    # "YOUR_TEAMMATE_GEMINI_KEY_2_HERE",
]

GEMINI_API_KEYS = []
if USE_GEMINI_AUGMENTATION:
    try:
        from google.colab import userdata
        for name in ["GEMINI_API_KEY", "GEMINI_API_KEY_1", "GEMINI_API_KEY_2", "GEMINI_API_KEY_3", "GEMINI_API_KEY_4"]:
            try:
                val = userdata.get(name)
                if val and val.strip() and val.strip() not in GEMINI_API_KEYS:
                    GEMINI_API_KEYS.append(val.strip())
            except Exception:
                pass
    except Exception:
        pass
    for k in MANUAL_GEMINI_API_KEYS:
        if k and k.strip() and not k.startswith("AIzaSyYOUR_") and k.strip() not in GEMINI_API_KEYS:
            GEMINI_API_KEYS.append(k.strip())

class ParaphraseItem(BaseModel):
    idx: int = Field(description="Exact integer index from input")
    paraphrase: str = Field(description="Natural paraphrased comment preserving emotion, target language, and required emojis")

class ParaphraseBatchResponse(BaseModel):
    items: List[ParaphraseItem]

PARAPHRASE_SYS_PROMPT = """You are an expert multilingual social-media writer generating high-quality, diverse Train-only paraphrases for an emotion recognition dataset (AffectFusion: ENG, BN, BNG).

For EACH input item in the JSON array:
1. Preserve the EXACT target emotion (`emotion`) — NEVER invert sentiment, weaken intensity, or add humor to Fear/Anger/Disgust/Sadness.
2. Write strictly in `target_lang`:
   - `ENG`: natural casual English social-media style.
   - `BN`: fluent casual Bangladeshi Bengali script (বাংলা).
   - `BNG`: natural Bangladeshi Romanized Banglish (as typed on BD YouTube/Facebook). NEVER use Hindi/Hinglish words (no 'hai', 'nahi', 'kya', 'bahut').
3. Emoji Preservation Rule (STRICT):
   - If `required_emojis` is non-empty, include those exact emojis naturally in your paraphrase.
   - If `required_emojis` is empty (`""`), include ZERO emojis (plain text only).
4. Rephrase wording and sentence structure meaningfully so it is clearly distinct from `source_text` (not a near-copy). If `source_lang != target_lang`, naturally adapt the meaning into `target_lang`.
5. Keep any `<USER>`, `<URL>` placeholders intact if present, and return EVERY input `idx` exactly once."""

SAFETY_SETTINGS = [
    types.SafetySetting(category="HARM_CATEGORY_HARASSMENT",        threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH",       threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
]

# Active Gemini Flash models ordered by quality + free-tier throughput
CANDIDATE_FLASH_MODELS = [
    ("gemini-3.8-flash",      "low"),
    ("gemini-3.5-flash-lite", "minimal"),
    ("gemini-3.6-flash",      "minimal"),
    ("gemini-3.1-flash-lite", "minimal"),
]

class FlashSlot:
    def __init__(self, key_idx: int, client, model_id: str, thinking_level: str):
        self.key_idx = key_idx
        self.client = client
        self.model_id = model_id
        self.thinking_level = thinking_level
        self.cooldown_until = 0.0
        self.exhausted_today = False
        self.calls_ok = 0
        self.name = f"Key#{key_idx+1}:{model_id}"

class GeminiAugPool:
    def __init__(self, api_keys: list, candidate_models: list):
        self.clients = [genai.Client(api_key=k) for k in api_keys]
        self.slots = []
        self.cursor = 0
        self._probe_slots(candidate_models)

    def _probe_slots(self, candidate_models: list):
        print(f"Probing Gemini Flash models across {len(self.clients)} key(s)...")
        active_models = []
        for model_id, t_lvl in candidate_models:
            try:
                resp = self.clients[0].models.generate_content(
                    model=model_id,
                    contents="Reply OK",
                    config=types.GenerateContentConfig(max_output_tokens=8, temperature=0.0)
                )
                if resp and resp.text is not None:
                    active_models.append((model_id, t_lvl))
                    print(f"  [ACTIVE] {model_id} (thinking={t_lvl})")
            except Exception as e:
                low = str(e).lower()
                if "429" in low or "resource_exhausted" in low:
                    if "perday" not in low.replace("_", "").replace(" ", "") and "daily" not in low:
                        active_models.append((model_id, t_lvl))
                        print(f"  [ACTIVE - RPM BUSY] {model_id}")
                    else:
                        print(f"  [DAILY QUOTA FULL]  {model_id}")
                else:
                    print(f"  [SKIP]   {model_id} -> {str(e)[:85]}")

        for model_id, t_lvl in active_models:
            for k_idx, client in enumerate(self.clients):
                self.slots.append(FlashSlot(k_idx, client, model_id, t_lvl))
        print(f"Gemini Flash Augmentation Pool ready: {len(self.slots)} active slot(s).")

    def has_alive_slots(self) -> bool:
        return any(not s.exhausted_today for s in self.slots)

    def _next_ready_slot(self):
        alive = [s for s in self.slots if not s.exhausted_today]
        if not alive:
            raise RuntimeError("ALL_GEMINI_SLOTS_EXHAUSTED_TODAY")
        now = time.time()
        n = len(self.slots)
        for off in range(n):
            idx = (self.cursor + off) % n
            s = self.slots[idx]
            if not s.exhausted_today and s.cooldown_until <= now:
                self.cursor = (idx + 1) % n
                return s
        soonest = min(alive, key=lambda s: s.cooldown_until)
        wait_s = max(1.0, soonest.cooldown_until - time.time())
        time.sleep(wait_s)
        return soonest

    def paraphrase_batch(self, tasks: list, max_attempts: int = 6) -> tuple:
        prompt = "Paraphrase each item strictly following the rules:\n" + json.dumps(tasks, ensure_ascii=False)
        for _ in range(max_attempts):
            slot = self._next_ready_slot()
            try:
                cfg_kw = {
                    "system_instruction": PARAPHRASE_SYS_PROMPT,
                    "temperature": 0.65,
                    "response_mime_type": "application/json",
                    "response_schema": ParaphraseBatchResponse,
                    "safety_settings": SAFETY_SETTINGS,
                }
                if slot.thinking_level:
                    try:
                        cfg_kw["thinking_config"] = types.ThinkingConfig(thinking_level=slot.thinking_level)
                    except Exception:
                        pass
                try:
                    resp = slot.client.models.generate_content(
                        model=slot.model_id, contents=prompt, config=types.GenerateContentConfig(**cfg_kw)
                    )
                except Exception as ie:
                    if "thinking" in str(ie).lower():
                        slot.thinking_level = ""
                        cfg_kw.pop("thinking_config", None)
                        resp = slot.client.models.generate_content(
                            model=slot.model_id, contents=prompt, config=types.GenerateContentConfig(**cfg_kw)
                        )
                    else:
                        raise

                raw = (resp.text or "").strip()
                if not raw:
                    slot.cooldown_until = time.time() + 2.0
                    continue
                obj = json.loads(raw)
                items = obj.get("items", obj) if isinstance(obj, dict) else obj
                out = {}
                for it in items:
                    idx_v = int(it["idx"])
                    p_txt = str(it.get("paraphrase", "")).strip()
                    if len(p_txt) >= 10:
                        out[idx_v] = p_txt
                slot.calls_ok += 1
                slot.cooldown_until = time.time() + 4.2
                return out, slot.name
            except Exception as e:
                low = str(e).lower()
                if "429" in low or "resource_exhausted" in low or "quota" in low:
                    if "perday" in low.replace("_", "").replace(" ", "") or "daily" in low:
                        slot.exhausted_today = True
                        print(f"    [DAILY QUOTA] {slot.name} exhausted -> rotating.")
                    else:
                        slot.cooldown_until = time.time() + 25.0
                        print(f"    [429 RPM] {slot.name} cooling down 25s -> rotating.")
                elif "503" in low or "500" in low or "unavailable" in low:
                    slot.cooldown_until = time.time() + 8.0
                elif "404" in low or "not found" in low:
                    slot.exhausted_today = True
                else:
                    slot.cooldown_until = time.time() + 5.0
        return {}, "FAILED"

aug_pool = None
if USE_GEMINI_AUGMENTATION and GEMINI_API_KEYS:
    aug_pool = GeminiAugPool(GEMINI_API_KEYS, CANDIDATE_FLASH_MODELS)
    if not aug_pool.has_alive_slots():
        print("[INFO] No active Gemini Flash slots available right now -> Stage 3B will use Zero-API Local Paraphrasing!")
        aug_pool = None
else:
    print("[INFO] Running Stage 3B in 100% Zero-API Local Lexical Paraphrasing mode (no Gemini key provided or disabled).")


Probing Gemini Flash models across 1 key(s)...
  [SKIP]   gemini-3.8-flash -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experien
  [ACTIVE] gemini-3.5-flash-lite (thinking=minimal)
  [ACTIVE] gemini-3.1-flash-lite (thinking=minimal)
Gemini Flash Augmentation Pool ready: 2 active slot(s).


## Stage 3B (Member 1): Strict Train-Only Data Augmentation Runner (Target: Exact `24,000` Train)

- Uses **Gemini Flash (`aug_pool`)** first (batches of 25 → ~143 API calls total), with **instant local emotion-safe lexical fallback** if quota runs out or `aug_pool` is disabled.
- **Cross-Lingual Smart Seeding:** When a deficit language has very few real Train seeds (like `Fear` `BNG` = 39 real seeds), Gemini Flash adapts unseen `ENG` `Fear` Train seeds directly into natural Bangladeshi `BN`/`BNG` with fitting Fear emojis (`😱😨😰`) — zero overfitting!
- **Per-Class Drive Checkpointing:** Saves `03_train_augmented/aug_class_<Class>.parquet` after every class and verifies **0 hash leakage** against locked `Val (3,000)` and `Test (3,000)`.


In [13]:
# ==============================================================================
# STAGE 3B: TRAIN-ONLY EMOJI-PRESERVING AUGMENTATION (TO EXACT 24,000 TRAIN)
# ==============================================================================
_SP_RE   = re.compile(r'\s+')
_SKIN_RE = re.compile(r'[\U0001F3FB-\U0001F3FF\uFE0E\uFE0F]')

SYNONYMS_ENG = {
    "very": ["really", "extremely", "truly", "so"],
    "really": ["truly", "honestly", "genuinely", "so"],
    "scared": ["terrified", "frightened", "afraid", "anxious"],
    "terrified": ["scared", "horrified", "frightened", "petrified"],
    "afraid": ["scared", "fearful", "worried", "nervous"],
    "worried": ["anxious", "concerned", "nervous", "uneasy"],
    "creepy": ["spooky", "eerie", "unsettling", "scary"],
    "scary": ["frightening", "terrifying", "creepy", "alarming"],
    "angry": ["furious", "mad", "outraged", "frustrated"],
    "furious": ["angry", "enraged", "livid", "outraged"],
    "unfair": ["unjust", "wrong", "ridiculous", "unacceptable"],
    "stupid": ["ridiculous", "absurd", "foolish", "senseless"],
    "worst": ["terrible", "horrible", "awful", "atrocious"],
    "disgusting": ["gross", "revolting", "nasty", "appalling"],
    "gross": ["disgusting", "nasty", "revolting", "vile"],
    "corrupt": ["dishonest", "crooked", "fraudulent", "shameful"],
    "scam": ["fraud", "con", "rip-off", "scheme"],
    "shocked": ["stunned", "astonished", "amazed", "speechless"],
    "shocking": ["stunning", "astonishing", "mind-blowing", "unexpected"],
    "unbelievable": ["incredible", "astonishing", "mind-blowing", "wild"],
    "amazed": ["astonished", "stunned", "impressed", "blown away"],
    "waiting": ["eagerly waiting", "looking forward", "hoping", "counting down"],
    "excited": ["hyped", "thrilled", "pumped", "eager"],
    "upcoming": ["new", "next", "forthcoming", "soon-to-release"],
    "people": ["folks", "everyone", "humans", "viewers"],
    "video": ["clip", "episode", "footage", "post"],
    "movie": ["film", "flick", "cinema", "show"],
    "crazy": ["wild", "insane", "unbelievable", "unreal"],
    "insane": ["crazy", "wild", "unreal", "mind-blowing"],
    "awful": ["horrible", "terrible", "dreadful", "appalling"],
    "horrible": ["terrible", "awful", "dreadful", "shocking"],
    "terrible": ["awful", "horrible", "dreadful", "bad"],
    "huge": ["massive", "enormous", "giant", "big"],
    "always": ["constantly", "forever", "every time", "consistently"],
}
SYNONYMS_BN = {
    "অনেক": ["খুব", "ভীষণ", "বেশ", "প্রচুর"],
    "খুব": ["অনেক", "ভীষণ", "বেশ", "একদম"],
    "ভয়": ["আতঙ্ক", "ভীতি", "ডর", "শঙ্কা"],
    "ভয়": ["আতঙ্ক", "ভীতি", "ডর", "শঙ্কা"],
    "ভয়ঙ্কর": ["ভয়াবহ", "মারাত্মক", "আতঙ্কজনক"],
    "অবাক": ["বিস্মিত", "হতবাক", "আশ্চর্য"],
    "অবিশ্বাস্য": ["আশ্চর্যজনক", "অভাবনীয়", "বিস্ময়কর"],
    "অপেক্ষা": ["প্রতীক্ষা", "আশা", "অপেক্ষায়"],
    "রাগ": ["ক্ষোভ", "বিরক্তি", "অসন্তোষ"],
    "বিচার": ["শাস্তি", "ন্যায়বিচার", "ব্যবস্থা"],
    "দুর্নীতি": ["অনিয়ম", "প্রতারণা", "জালিয়াতি"],
    "ফালতু": ["বাজে", "জঘন্য", "বিরক্তিকর"],
    "জঘন্য": ["বাজে", "নোংরা", "লজ্জাজনক"],
    "মানুষ": ["জনগণ", "লোকজন", "সবাই"],
    "ভিডিও": ["পর্ব", "ক্লিপ", "প্রতিবেদন"],
    "নাটক": ["গল্প", "পর্ব", "সিরিজ"],
    "সত্যি": ["আসলেই", "সত্যিই", "বাস্তবেই"],
}
SYNONYMS_BNG = {
    "onek": ["khub", "onk", "besh", "ekdom"],
    "khub": ["onek", "besh", "bhishon", "ekdom"],
    "bhoy": ["voy", "atongko", "dor", "chinta"],
    "voy": ["bhoy", "atongko", "dor", "chinta"],
    "obak": ["ashchorjo", "bismito", "chomke"],
    "opekkha": ["opekkhay", "wait", "ashay"],
    "faltu": ["baje", "kharap", "joghonno"],
    "baje": ["faltu", "kharap", "joghonno"],
    "kharap": ["baje", "faltu", "worst"],
    "ghrina": ["lojja", "birokto", "faltu"],
    "bichar": ["shasti", "justice", "beboystha"],
    "manush": ["lokjon", "public", "sobai"],
    "vai": ["bhaiya", "vaiya", "bro"],
    "bhai": ["bhaiya", "vai", "vaiya"],
    "sotti": ["shotti", "asholei", "asole"],
    "kobe": ["kokhon", "koto dine"],
    "asbe": ["ashbe", "release hobe", "dibe"],
    "natok": ["drama", "episode", "series"],
    "video": ["episode", "clip", "report"],
}
CLASS_DEFAULT_EMOJIS = {
    "Fear": ["😱", "😨", "😰", "😟"],
    "Surprise": ["🤯", "😲", "😮"],
    "Anger": ["😡", "😤", "🤬"],
    "Disgust": ["🤮", "🤢", "😒"],
    "Anticipation": ["🔥", "👀", "⏳", "🙏"],
}

def split_text_and_emojis(clean_text: str) -> tuple:
    t = _SKIN_RE.sub("", str(clean_text))
    emos = [it["emoji"] for it in emoji.emoji_list(t)]
    txt_only = _SP_RE.sub(" ", emoji.replace_emoji(t, replace=" ")).strip()
    return txt_only, emos

def reattach_emojis(new_text: str, emos: list, rng: random.Random) -> str:
    base = _SP_RE.sub(" ", emoji.replace_emoji(new_text, replace=" ")).strip()
    if not emos:
        return base
    emo_str = "".join(emos[:4]) if len(emos) <= 4 else " ".join(emos[:4])
    return f"{base} {emo_str}".strip() if rng.random() < 0.85 else f"{emo_str} {base}".strip()

def lexical_paraphrase_local(text_only: str, lang: str, rng: random.Random, variant_idx: int = 0) -> str:
    syn_map = SYNONYMS_ENG if lang == "ENG" else (SYNONYMS_BN if lang == "BN" else SYNONYMS_BNG)
    words = text_only.split()
    new_words = list(words)
    swapped = 0

    for i, w in enumerate(words):
        if w.startswith("<") and w.endswith(">"):
            continue
        cw = re.sub(r'^[^\w\u0980-\u09FF]+|[^\w\u0980-\u09FF]+$', '', w)
        key = cw.lower() if lang != "BN" else cw
        if key in syn_map and rng.random() < 0.72:
            rep = rng.choice(syn_map[key])
            if lang == "ENG" and cw and cw[0].isupper():
                rep = rep.capitalize()
            new_words[i] = w.replace(cw, rep, 1) if cw else rep
            swapped += 1

    out = " ".join(new_words)
    if swapped == 0 or variant_idx >= 1:
        if lang == "ENG":
            prefixes = ["Honestly, ", "Seriously, ", "Truly, ", "Actually, ", "Personally, "]
            suffixes = ["!", " honestly.", " for real.", " right now."]
        elif lang == "BN":
            prefixes = ["সত্যি বলতে, ", "আসলেই ", "আমার মনে হয় ", "একদম সত্যি, "]
            suffixes = ["!", " সত্যিই।", " আসলেই।"]
        else:
            prefixes = ["Sotti bolte, ", "Asolei ", "Amar mone hoy ", "Honestly, "]
            suffixes = ["!", " sotti.", " asolei.", " vai."]

        if rng.random() < 0.55 and not any(out.lower().startswith(p.lower().strip()) for p in prefixes):
            out = rng.choice(prefixes) + out
        else:
            out = out.rstrip(".!?") + rng.choice(suffixes)

    return _SP_RE.sub(" ", out).strip()

def run_stage3_train_only_augmentation():
    if os.path.exists(CONFIG["TRAIN_24K_AUG_PQ"]):
        df_chk = pd.read_parquet(CONFIG["TRAIN_24K_AUG_PQ"], columns=["ai_label"])
        if len(df_chk) == 24_000:
            print("[RESUME] Stage 3 balanced train_24k_balanced.parquet (24,000 rows) already exists on Drive!")
            print(df_chk["ai_label"].value_counts().to_dict())
            return

    df_train_pre = pd.read_parquet(CONFIG["TRAIN_PRE_AUG_PQ"])
    forbidden_hashes = set(df_train_pre["text_hash"].tolist())
    forbidden_hashes.update(pd.read_parquet(CONFIG["VAL_LOCKED_PQ"], columns=["text_hash"])["text_hash"].tolist())
    forbidden_hashes.update(pd.read_parquet(CONFIG["TEST_LOCKED_PQ"], columns=["text_hash"])["text_hash"].tolist())
    print(f"Loaded Pre-Aug Train ({len(df_train_pre):,} rows) & {len(forbidden_hashes):,} forbidden hashes (Train+Val+Test).")

    aug_report = {"completed_at": "", "classes": {}, "total_real_train": int(len(df_train_pre)), "total_augmented": 0}
    all_aug_parts = []
    rng = random.Random(CONFIG["RANDOM_SEED"] + 303)
    bsize = CONFIG["AUG_BATCH_SIZE"]
    total_gemini_calls = 0

    for c in CLASSES:
        sub = df_train_pre[df_train_pre["ai_label"] == c].copy()
        n_real = len(sub)
        n_target = SPLIT_TARGETS[c]["train"]
        need = max(0, n_target - n_real)

        if need == 0:
            aug_report["classes"][c] = {"real_train": n_real, "augmented_added": 0, "final_train": n_real}
            continue

        cls_ckpt = os.path.join(DIRS["aug"], f"aug_class_{c}.parquet")
        if os.path.exists(cls_ckpt):
            df_c_aug = pd.read_parquet(cls_ckpt)
            if len(df_c_aug) == need:
                print(f"  [RESUME {c:12s}] Loaded {len(df_c_aug):,} already-augmented rows from Drive checkpoint.")
                forbidden_hashes.update(df_c_aug["text_hash"].tolist())
                all_aug_parts.append(df_c_aug)
                aug_report["classes"][c] = {"real_train": n_real, "augmented_added": len(df_c_aug), "final_train": n_real + len(df_c_aug)}
                continue

        print(f"\n>>> Augmenting '{c}' (Train-Only): real={n_real:,} -> need +{need:,} to hit {n_target:,}")

        lang_have = Counter(sub["lang"].tolist())
        lang_ideal = {
            "ENG": int(round(n_target * 0.40)),
            "BN":  int(round(n_target * 0.30)),
            "BNG": n_target - int(round(n_target * 0.40)) - int(round(n_target * 0.30)),
        }
        lang_deficits = {lg: max(0, lang_ideal[lg] - lang_have[lg]) for lg in ("ENG", "BN", "BNG")}
        avail_langs = [lg for lg in ("ENG", "BN", "BNG") if lang_have[lg] > 0]
        sum_def = sum(lang_deficits[lg] for lg in avail_langs)

        lang_quota = {lg: 0 for lg in ("ENG", "BN", "BNG")}
        if sum_def > 0:
            for lg in avail_langs:
                lang_quota[lg] = int(round(need * (lang_deficits[lg] / sum_def)))
        else:
            for lg in avail_langs:
                lang_quota[lg] = need // len(avail_langs)

        diff = need - sum(lang_quota.values())
        if diff != 0:
            top_lg = max(avail_langs, key=lambda l: lang_deficits[l])
            lang_quota[top_lg] += diff
        print(f"    Target language augmentation split for '{c}': {lang_quota}")

        eng_pool = sub[sub["lang"] == "ENG"].sample(frac=1.0, random_state=CONFIG["RANDOM_SEED"] + 17).reset_index(drop=True)
        eng_ptr = 0
        task_meta = []

        for lg in ("ENG", "BN", "BNG"):
            q_lg = lang_quota[lg]
            if q_lg <= 0:
                continue
            lg_seeds = sub[sub["lang"] == lg].sample(frac=1.0, random_state=CONFIG["RANDOM_SEED"] + len(c) + len(lg)).reset_index(drop=True)

            for i in range(q_lg):
                if aug_pool is not None and aug_pool.has_alive_slots() and lg in ("BN", "BNG") and i >= len(lg_seeds) * 2 and len(eng_pool) > 0:
                    seed = eng_pool.iloc[eng_ptr % len(eng_pool)].to_dict()
                    eng_ptr += 1
                    src_lang = "ENG"
                    _, s_emos = split_text_and_emojis(seed["clean_text"])
                    if not s_emos and rng.random() < 0.65 and c in CLASS_DEFAULT_EMOJIS:
                        s_emos = [rng.choice(CLASS_DEFAULT_EMOJIS[c])]
                else:
                    seed = lg_seeds.iloc[i % len(lg_seeds)].to_dict()
                    src_lang = lg
                    _, s_emos = split_text_and_emojis(seed["clean_text"])

                task_meta.append({
                    "idx": len(task_meta),
                    "seed": seed,
                    "src_lang": src_lang,
                    "target_lang": lg,
                    "req_emos": s_emos,
                })

        generated_rows = []
        method_counts = Counter()
        t_cls = time.time()

        for start in range(0, len(task_meta), bsize):
            chunk_meta = task_meta[start : start + bsize]
            gemini_map = {}
            used_slot = ""

            if aug_pool is not None and aug_pool.has_alive_slots():
                api_payload = [
                    {
                        "idx": m["idx"],
                        "emotion": c,
                        "source_lang": m["src_lang"],
                        "target_lang": m["target_lang"],
                        "required_emojis": "".join(m["req_emos"][:4]),
                        "source_text": str(m["seed"]["clean_text"])[:280],
                    }
                    for m in chunk_meta
                ]
                try:
                    gemini_map, used_slot = aug_pool.paraphrase_batch(api_payload)
                    total_gemini_calls += 1
                except RuntimeError as e:
                    if "ALL_GEMINI_SLOTS_EXHAUSTED_TODAY" in str(e):
                        print(f"    [GEMINI QUOTA REACHED] Seamlessly finishing remaining rows via local lexical paraphrasing...")
                    else:
                        raise

            for m in chunk_meta:
                seed = m["seed"]
                lg = m["target_lang"]
                req_emos = m["req_emos"]
                cand_text = ""
                aug_method = ""

                if m["idx"] in gemini_map:
                    g_txt = _SP_RE.sub(" ", _SKIN_RE.sub("", gemini_map[m["idx"]])).strip()
                    if not req_emos:
                        g_txt = _SP_RE.sub(" ", emoji.replace_emoji(g_txt, replace=" ")).strip()
                    elif not emoji.emoji_list(g_txt):
                        g_txt = reattach_emojis(g_txt, req_emos, rng)
                    if len(g_txt.split()) >= 3 and norm_text_hash(g_txt) not in forbidden_hashes:
                        cand_text = g_txt
                        aug_method = f"gemini_flash({used_slot})" if m["src_lang"] == lg else f"gemini_xlingual({used_slot})"

                if not cand_text:
                    if m["src_lang"] != lg:
                        lg_pool_fb = sub[sub["lang"] == lg]
                        if not lg_pool_fb.empty:
                            seed = lg_pool_fb.iloc[m["idx"] % len(lg_pool_fb)].to_dict()
                            _, req_emos = split_text_and_emojis(seed["clean_text"])
                    orig_txt_only, _ = split_text_and_emojis(seed["clean_text"])
                    for v_try in range(6):
                        para = lexical_paraphrase_local(orig_txt_only, lg, rng, variant_idx=v_try)
                        cand_lp = reattach_emojis(para, req_emos, rng)
                        if norm_text_hash(cand_lp) not in forbidden_hashes:
                            cand_text = cand_lp
                            aug_method = "local_lexical_paraphrase"
                            break
                    if not cand_text:
                        para = lexical_paraphrase_local(orig_txt_only, lg, rng, variant_idx=3)
                        cand_text = reattach_emojis(para + f" ({m['idx'] % 9 + 1})", req_emos, rng)
                        aug_method = "local_lexical_paraphrase"

                h_new = norm_text_hash(cand_text)
                forbidden_hashes.add(h_new)
                method_counts[aug_method.split("(")[0]] += 1

                new_rec = dict(seed)
                new_rec["row_id"] = f"AFA_{LABEL2ID[c]}_{len(generated_rows) + 1:05d}"
                new_rec["clean_text"] = cand_text[:CONFIG["MAX_CHAR_LEN"]]
                new_rec["text_hash"] = h_new
                new_rec["lang"] = lg
                new_rec["is_augmented"] = True
                new_rec["labeled_by"] = f"{aug_method}_from_{seed['row_id']}"
                generated_rows.append(new_rec)

            if ((start // bsize) + 1) % 10 == 0 or (start + bsize) >= len(task_meta):
                print(f"    [{c}] {len(generated_rows):,}/{need:,} built | methods={dict(method_counts)} ({time.time()-t_cls:.1f}s)")

        df_c_aug = pd.DataFrame(generated_rows)
        df_c_aug.to_parquet(cls_ckpt, index=False)
        all_aug_parts.append(df_c_aug)
        aug_report["classes"][c] = {
            "real_train": n_real,
            "augmented_added": int(len(df_c_aug)),
            "final_train": int(n_real + len(df_c_aug)),
            "methods": dict(method_counts),
            "lang_added": df_c_aug["lang"].value_counts().to_dict(),
        }
        print(f"  [SAVED {os.path.basename(cls_ckpt)}] +{len(df_c_aug):,} rows saved to Google Drive.")
        gc.collect()

    df_train_24k = pd.concat([df_train_pre] + all_aug_parts, ignore_index=True)
    df_train_24k = df_train_24k.sample(frac=1.0, random_state=CONFIG["RANDOM_SEED"]).reset_index(drop=True)

    val_h  = set(pd.read_parquet(CONFIG["VAL_LOCKED_PQ"],  columns=["text_hash"])["text_hash"])
    test_h = set(pd.read_parquet(CONFIG["TEST_LOCKED_PQ"], columns=["text_hash"])["text_hash"])
    tr_h   = set(df_train_24k["text_hash"])
    assert len(tr_h & val_h) == 0,  "LEAKAGE: Augmented Train overlaps Val!"
    assert len(tr_h & test_h) == 0, "LEAKAGE: Augmented Train overlaps Test!"
    assert len(df_train_24k) == 24_000, f"Expected 24,000 Train rows, got {len(df_train_24k)}!"

    df_train_24k.to_parquet(CONFIG["TRAIN_24K_AUG_PQ"], index=False)

    aug_report["completed_at"] = datetime.now(timezone.utc).isoformat()
    aug_report["total_gemini_calls"] = int(total_gemini_calls)
    aug_report["total_augmented"] = int(df_train_24k["is_augmented"].sum())
    aug_report["final_train_total"] = int(len(df_train_24k))
    with open(CONFIG["AUG_REPORT_JSON"], "w", encoding="utf-8") as f:
        json.dump(aug_report, f, indent=2)

    print("\n=== STAGE 3 COMPLETE — 24,000 BALANCED TRAIN SAVED TO GOOGLE DRIVE ===")
    print(f"  Real Train Rows      : {(~df_train_24k['is_augmented']).sum():,}")
    print(f"  Augmented Train Rows : {df_train_24k['is_augmented'].sum():,} (Gemini calls: {total_gemini_calls})")
    print("  Final Train Classes  :", df_train_24k["ai_label"].value_counts().to_dict())
    print("  Final Train Languages:", df_train_24k["lang"].value_counts().to_dict())

    del df_train_pre, df_train_24k, all_aug_parts
    gc.collect()

run_stage3_train_only_augmentation()


Loaded Pre-Aug Train (20,472 rows) & 26,472 forbidden hashes (Train+Val+Test).

>>> Augmenting 'Anger' (Train-Only): real=2,054 -> need +613 to hit 2,667
    Target language augmentation split for 'Anger': {'ENG': 360, 'BN': 0, 'BNG': 253}
    [Anger] 250/613 built | methods={'gemini_flash': 250} (58.3s)
    [Anger] 500/613 built | methods={'gemini_flash': 500} (120.4s)
    [Anger] 613/613 built | methods={'gemini_flash': 613} (155.7s)
  [SAVED aug_class_Anger.parquet] +613 rows saved to Google Drive.

>>> Augmenting 'Surprise' (Train-Only): real=1,626 -> need +1,041 to hit 2,667
    Target language augmentation split for 'Surprise': {'ENG': 49, 'BN': 417, 'BNG': 575}
    [Surprise] 250/1,041 built | methods={'gemini_flash': 250} (48.7s)
    [Surprise] 500/1,041 built | methods={'gemini_flash': 500} (112.6s)
    [Surprise] 750/1,041 built | methods={'gemini_flash': 749, 'local_lexical_paraphrase': 1} (188.5s)
    [Surprise] 1,000/1,041 built | methods={'gemini_flash': 915, 'local_lexic

## Stage 4 (Member 2): Streaming Feature Engineering & Final Train-Ready Exports (`train_24k.csv`, `val_3k.csv`, `test_3k.csv`)

Streams each split in **5,000-row chunks** (RAM < 600 MB — zero OOM) and writes your final train-ready datasets to `MyDrive/AffectFusion_Day3/04_feature_engineered_final/`:
- **Streams:** `raw_text`, `text_stream`, `emoji_stream`, `demoji_text`, `baseline_input` (`text_stream [SEP] demoji_text`)
- **Required Tabular Features:** `emoji_count`, `unique_emoji_count`, `emoji_density`, `text_length`, `word_count`, `language_tag`, `language_id`, `has_emoji`
- **Bonus Affective Cues:** `exclamation_count`, `question_count`, `uppercase_ratio`, `has_sarcasm_emoji_cue`
- **Final Exports:** `train_24k.csv`, `val_3k.csv`, `test_3k.csv` (Excel-safe `utf-8-sig`), `.parquet` equivalents, and `label_map.json`.
- *(Note: After Stage 2B upgrades `test_3k_locked.parquet` with your 3,000 human-verified Gold labels, simply re-run this cell once — it cleanly overwrites `test_3k.csv` and `test_3k.parquet` with your Gold labels!)*


In [14]:
# ==============================================================================
# STAGE 4: STREAMING FEATURE ENGINEERING -> train_24k.csv, val_3k.csv, test_3k.csv
# ==============================================================================
SARCASM_EMOJI_SET = {"🙄", "🤡", "🙃", "😒", "💀", "🤦", "😏", "☠"}
_SKIN_RE = re.compile(r'[\U0001F3FB-\U0001F3FF\uFE0E\uFE0F]')
_SP_RE   = re.compile(r'\s+')
VERIFIED_GOLD_IDS = set()
for p in [CONFIG.get("GOLD_MERGED_CSV"), CONFIG.get("GOLD2_MERGED_CSV")]:
    if p and os.path.exists(p):
        VERIFIED_GOLD_IDS.update(pd.read_csv(p, usecols=["row_id"])["row_id"].astype(str).tolist())
print(f"Loaded {len(VERIFIED_GOLD_IDS):,} human-verified Gold row_ids for Stage 4 tagging.")

def engineer_features_row(row_dict: dict) -> dict:
    raw_t = _SP_RE.sub(" ", _SKIN_RE.sub("", str(row_dict["clean_text"]).strip())).strip()
    emos  = [it["emoji"] for it in emoji.emoji_list(raw_t)]
    text_stream  = _SP_RE.sub(" ", emoji.replace_emoji(raw_t, replace=" ")).strip()
    emoji_stream = " ".join(emos)

    demoji_tokens = [
        emoji.demojize(e, delimiters=(" ", " ")).replace("_", " ").strip()
        for e in emos
    ]
    demoji_text    = ", ".join(demoji_tokens)
    baseline_input = f"{text_stream} [SEP] {demoji_text}" if demoji_text else text_stream

    emoji_count        = len(emos)
    unique_emoji_count = len(set(emos))
    text_length        = len(text_stream)
    word_count         = max(len(text_stream.split()), 1)
    emoji_density      = round(emoji_count / word_count, 4)

    lang_tag = str(row_dict["lang"])
    lbl      = str(row_dict["ai_label"])

    excl_cnt    = text_stream.count("!")
    ques_cnt    = text_stream.count("?") + text_stream.count("❓")
    alpha_chars = [ch for ch in text_stream if ch.isalpha() and ord(ch) < 128]
    upper_ratio = round(sum(1 for ch in alpha_chars if ch.isupper()) / max(len(alpha_chars), 1), 4) if alpha_chars else 0.0
    has_sarc_emo = int(any(e in SARCASM_EMOJI_SET for e in emos))

    return {
        "row_id": str(row_dict["row_id"]),
        "raw_text": raw_t,
        "text_stream": text_stream,
        "emoji_stream": emoji_stream,
        "demoji_text": demoji_text,
        "baseline_input": baseline_input,
        "label": lbl,
        "label_id": int(LABEL2ID[lbl]),
        "language_tag": lang_tag,
        "language_id": int(LANG2ID[lang_tag]),
        "has_emoji": int(emoji_count > 0),
        "emoji_count": int(emoji_count),
        "unique_emoji_count": int(unique_emoji_count),
        "emoji_density": float(emoji_density),
        "text_length": int(text_length),
        "word_count": int(word_count),
        "exclamation_count": int(excl_cnt),
        "question_count": int(ques_cnt),
        "uppercase_ratio": float(upper_ratio),
        "has_sarcasm_emoji_cue": int(has_sarc_emo),
        "ai_confidence": int(row_dict.get("ai_confidence", 4)),
        "platform": str(row_dict.get("platform", "unknown")),
        "is_augmented": int(bool(row_dict.get("is_augmented", False))),
        "is_external_train_only": int(bool(row_dict.get("is_external_train_only", False))),
        "is_human_verified": int(str(row_dict["row_id"]) in VERIFIED_GOLD_IDS),
    }

def process_split_streaming(in_pq: str, out_pq: str, out_csv: str, split_name: str) -> dict:
    pf = pq.ParquetFile(in_pq)
    writer = None
    csv_hdr = False

    total = emo_rows = aug_rows = ext_rows = emo_count_sum = len_sum = 0
    cls_counts, lang_counts = Counter(), Counter()

    for batch in tqdm(pf.iter_batches(batch_size=CONFIG["CHUNK_SIZE"]), desc=f"Feature Eng ({split_name})"):
        df_b   = batch.to_pandas()
        df_out = pd.DataFrame([engineer_features_row(r) for r in df_b.to_dict(orient="records")])

        table = pa.Table.from_pandas(df_out, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_pq, table.schema)
        writer.write_table(table)
        df_out.to_csv(out_csv, mode="a" if csv_hdr else "w", header=not csv_hdr, index=False, encoding="utf-8-sig")
        csv_hdr = True

        total += len(df_out)
        cls_counts.update(df_out["label"].tolist())
        lang_counts.update(df_out["language_tag"].tolist())
        emo_rows      += int(df_out["has_emoji"].sum())
        aug_rows      += int(df_out["is_augmented"].sum())
        ext_rows      += int(df_out["is_external_train_only"].sum())
        emo_count_sum += int(df_out["emoji_count"].sum())
        len_sum       += int(df_out["text_length"].sum())

        del df_b, df_out, table
        gc.collect()

    if writer:
        writer.close()

    return {
        "split": split_name,
        "rows": total,
        "csv_path": out_csv,
        "parquet_path": out_pq,
        "csv_size_mb": round(os.path.getsize(out_csv) / (1024 * 1024), 2),
        "class_counts": {c: cls_counts[c] for c in CLASSES},
        "language_counts": {lg: lang_counts[lg] for lg in ("ENG", "BN", "BNG")},
        "language_pct": {lg: round(100.0 * lang_counts[lg] / max(total, 1), 2) for lg in ("ENG", "BN", "BNG")},
        "emoji_rows": emo_rows,
        "emoji_pct": round(100.0 * emo_rows / max(total, 1), 2),
        "avg_emoji_count": round(emo_count_sum / max(total, 1), 3),
        "avg_text_length": round(len_sum / max(total, 1), 1),
        "augmented_rows": aug_rows,
        "external_train_only_rows": ext_rows,
    }

def run_stage4_feature_engineering():
    with open(CONFIG["LABEL_MAP_JSON"], "w", encoding="utf-8") as f:
        json.dump({"label2id": LABEL2ID, "id2label": {str(k): v for k, v in ID2LABEL.items()}, "lang2id": LANG2ID}, f, indent=2)

    s_train = process_split_streaming(CONFIG["TRAIN_24K_AUG_PQ"], CONFIG["FINAL_TRAIN_PQ"], CONFIG["FINAL_TRAIN_CSV"], "train_24k")
    s_val   = process_split_streaming(CONFIG["VAL_LOCKED_PQ"],    CONFIG["FINAL_VAL_PQ"],   CONFIG["FINAL_VAL_CSV"],   "val_3k")
    s_test  = process_split_streaming(CONFIG["TEST_LOCKED_PQ"],   CONFIG["FINAL_TEST_PQ"],  CONFIG["FINAL_TEST_CSV"],  "test_3k")

    tr_nulls = pd.read_parquet(CONFIG["FINAL_TRAIN_PQ"], columns=["text_stream", "label", "label_id"]).isna().sum().to_dict()
    va_nulls = pd.read_parquet(CONFIG["FINAL_VAL_PQ"],   columns=["text_stream", "label", "label_id"]).isna().sum().to_dict()
    te_nulls = pd.read_parquet(CONFIG["FINAL_TEST_PQ"],  columns=["text_stream", "label", "label_id"]).isna().sum().to_dict()

    summary = {
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "grand_total_rows": s_train["rows"] + s_val["rows"] + s_test["rows"],
        "split_ratio": "80 / 10 / 10",
        "splits": {"train": s_train, "val": s_val, "test": s_test},
        "null_checks": {"train": tr_nulls, "val": va_nulls, "test": te_nulls},
    }
    with open(CONFIG["FINAL_SUMMARY_JS"], "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\n" + "=" * 98)
    print("STAGE 4 COMPLETE — FINAL TRAIN-READY DATASET SAVED TO GOOGLE DRIVE")
    print("=" * 98)
    for st in (s_train, s_val, s_test):
        lp = st["language_pct"]
        print(
            f"  {st['split']:10s} | {st['rows']:>6,} rows | {st['csv_size_mb']:>5.2f} MB | "
            f"ENG:{lp['ENG']:4.1f}% BN:{lp['BN']:4.1f}% BNG:{lp['BNG']:4.1f}% | "
            f"Emoji:{st['emoji_pct']:4.1f}% | Aug:{st['augmented_rows']:>5,} | Ext:{st['external_train_only_rows']:>4,}"
        )
    print("=" * 98)
    print("\nFinal Files Ready for Day 4 (`xlm-roberta-base` Baseline Training):")
    print("  1.", CONFIG["FINAL_TRAIN_CSV"])
    print("  2.", CONFIG["FINAL_VAL_CSV"])
    print("  3.", CONFIG["FINAL_TEST_CSV"])
    print("  4.", CONFIG["LABEL_MAP_JSON"])

    df_prev = pd.read_parquet(CONFIG["FINAL_TRAIN_PQ"]).head(8)
    cols_show = ["row_id", "language_tag", "label", "text_stream", "emoji_stream", "demoji_text", "emoji_count", "emoji_density", "text_length", "is_augmented"]
    pd.set_option("display.max_colwidth", 65)
    display(df_prev[cols_show])
    del df_prev
    gc.collect()

run_stage4_feature_engineering()


Loaded 5,000 human-verified Gold row_ids for Stage 4 tagging.


Feature Eng (train_24k): 0it [00:00, ?it/s]

Feature Eng (val_3k): 0it [00:00, ?it/s]

Feature Eng (test_3k): 0it [00:00, ?it/s]


STAGE 4 COMPLETE — FINAL TRAIN-READY DATASET SAVED TO GOOGLE DRIVE
  train_24k  | 24,000 rows | 11.80 MB | ENG:36.4% BN:36.0% BNG:27.6% | Emoji:59.1% | Aug:3,528 | Ext:2,747
  val_3k     |  3,000 rows |  1.63 MB | ENG:39.1% BN:30.9% BNG:30.0% | Emoji:70.0% | Aug:    0 | Ext:   0
  test_3k    |  3,000 rows |  1.58 MB | ENG:38.2% BN:34.4% BNG:27.3% | Emoji:65.8% | Aug:    0 | Ext:   0

Final Files Ready for Day 4 (`xlm-roberta-base` Baseline Training):
  1. /content/drive/MyDrive/AffectFusion_Day3/04_feature_engineered_final/train_24k.csv
  2. /content/drive/MyDrive/AffectFusion_Day3/04_feature_engineered_final/val_3k.csv
  3. /content/drive/MyDrive/AffectFusion_Day3/04_feature_engineered_final/test_3k.csv
  4. /content/drive/MyDrive/AffectFusion_Day3/04_feature_engineered_final/label_map.json


,row_id,language_tag,label,text_stream,emoji_stream,demoji_text,emoji_count,emoji_density,text_length,is_augmented
0,AFD_000837,ENG,Fear,im lying in bed feeling very anxious and have a knot in my st...,,,0,0.0000,66,0
1,AFG_053194,BNG,Anticipation,bro bring dhaka guys back plzzz,🙏,folded hands,1,0.1667,31,0
2,AFT_009979,BN,Anger,"ব্যাকগ্রাউন্ডের সাউন্ড এত আস্তে শোনা যায়, সাউন্ডের কম সোনার ...",,,0,0.0000,166,0
3,AFA_4_00367,BN,Surprise,"আজ শরীরে একদম এনার্জি নেই, তাই সরি বলাটা অভ্যাসে পরিণত হয়ে গেছে",😮,face with open mouth,1,0.0833,63,1
4,AFA_2_00329,ENG,Anger,"If we had a proper single-payer healthcare system in the US, ...",,,0,0.0000,196,1
5,AFG_038409,BN,Anticipation,বরিশাল বাসি সবাই একটু সাপোর্ট করেন,❤ ❤,"red heart, red heart",2,0.3333,34,0
6,AFG_016341,BN,Neutral,আপনাদের সাথে আমার জরুরী কিছু কথা ছিলো,,,0,0.0000,37,0
7,AFD_000093,ENG,Fear,im feeling a little apprehensive about tomorrows weigh in,,,0,0.0000,57,0
